# Outstanding tasks

| Owner | Task | What needs to be done |
|---|---|---|
| C, Nico | #5 Other windows | Run the event study on [-1, +1] and [-5, +5] as well as the [-10, +10] baseline. |
| C, Nico | Significance tests | Test whether CARs differ from zero, for negative and positive events: Boehmer et al. t-statistic, Cowan's sign test and a t-test clustered by date. |
| C, Nico | Robustness | Fama-French CARs, other overlap gaps, a β = 1 model, dropping stale-price banks, and dropping events with another campaign inside the window. |
| Alissa | #6 Do NGO campaigns matter? | In `Task6.ipynb`: split CARs by COP21, event type, prominence and sentiment, run one regression as the check on the splits, and plot the main figure. |
| D | #1 or #7 Add-on | Build one variable per campaign (media coverage or political preferences), keyed on `uid_archive`. It becomes one more split in #6. |
| D | Slides | Write the narrative and the four slides, once #6 has results. |

In [ ]:
import zipfile, io
import pandas as pd
import numpy as np
from pathlib import Path
import sys
import re
import html
import statsmodels.formula.api as smf

In [ ]:
DATA = Path("data") if Path("data").exists() else Path(".")   # data/ folder if there is one, otherwise the project folder
OUT = DATA / "handoff"                                        # files passed between A, B, C and D
OUT.mkdir(exist_ok=True)

if not (DATA / "sigwatch_data").exists():
    for z in ["ngo_data_bocconi.zip", "banks_data_bocconi.zip"]:
        zipfile.ZipFile(DATA / z).extractall(DATA)

---
# Part A: NGO data (owner: A, Cecilia)
Load and append the Sigwatch years, filter to US/UK/EU banks, describe the campaigns, and build the name→ISIN lookup.

**Handoff to C and D:** `handoff/events_banks.csv`, `handoff/campaigns_banks.csv`, `handoff/lookup_banks.csv`.

## A1. Load and append the Sigwatch years

In [ ]:
# Sigwatch
files = sorted((DATA / "sigwatch_data").glob("*.dta"))

sig = pd.concat(
    [pd.read_stata(f).assign(year_file=int(f.name[:4])) for f in files],
    ignore_index=True,
)
print(sig.shape)

In [ ]:
#Datastream
ri_sheets = pd.read_excel(DATA / "banks_data_bocconi" / "banks_ri.xlsm", sheet_name=None)
ri_sheets.pop("REQUEST_TABLE")

print(ri_sheets.keys())
print(ri_sheets["US"].shape)

In [ ]:
longs = []

for k, v in ri_sheets.items():
    long = v.melt(id_vars="Name", var_name="bank", value_name="ri")
    long = long.rename(columns={"Name": "date"})
    long["country"] = k
    longs.append(long)

ri = pd.concat(longs, ignore_index=True)

In [ ]:
print(ri.shape)
print(ri["country"].value_counts())
print(ri["bank"].nunique())

In [ ]:
print(f"{sig['uid_archive'].nunique():,} events, {len(sig):,} rows")
print(f"{sig['company'].nunique():,} companies in total") #nbr of unique company_name -> Number of companies
print(f"Average : {len(sig) / sig['uid_archive'].nunique():.1f} companies per event")

**Largest event:**

In [ ]:
top_uid = sig["uid_archive"].value_counts().index[0]
top = sig[sig["uid_archive"] == top_uid]
print(f"Largest event: {len(top)} firms named by {top['ngo_name1'].iloc[0]} on {top['date'].iloc[0]}")

Largest event. The event with the most targets is a single communication by ActionAid UK on 13 May 2013, naming 72 companies at once (BP, Shell, Tesco, AstraZeneca, …).

This matters for the event study: an NGO communication that names dozens of firms is unlikely to be firm-specific news, and any common price move that day would be absorbed by the market factor. We therefore compute n_targets (number of firms named in the same event) and will either drop or analyse separately the mass events above a threshold.

In [ ]:
counts = sig["uid_archive"].value_counts()          #number of firms per event
sig["n_targets"] = sig["uid_archive"].map(counts)   #attach it to every (event, firm) row

In [ ]:
events = sig.drop_duplicates("uid_archive")

print("n_targets, per (event, firm) row:")
print(sig["n_targets"].describe().round(2))

print("\nn_targets, per event:")
print(events["n_targets"].describe().round(2))

share_single = (events["n_targets"] == 1).mean()
print(f"\n{share_single:.1%} of events target a single company")

## A2. Clean Sigwatch and filter to US/UK/EU banks

In [ ]:
#Basic cleaning
sig["date"] = pd.to_datetime(sig["date"])                                    # dates are stored as text
for c in ["isin_1", "isin_2", "isin_3"]:
    sig[c] = sig[c].replace("", np.nan)                                      # empty strings -> missing
sig["parent_country"] = sig["company_parent_country"].replace({"USA": "US"}) # same country, two spellings

In [ ]:
# Names of the banks we have prices for (Datastream, incl. Swiss/Nordic file)

# Read the column headers of every country sheet and strip the " - TOT RETURN IND" suffix to get the bank name
def ds_names(file):
    sheets = pd.read_excel(DATA / "banks_data_bocconi" / file, sheet_name=None, nrows=1)
    sheets.pop("REQUEST_TABLE")
    return {c.rsplit(" - ", 1)[0].strip().lower() for v in sheets.values() for c in v.columns if c != "Name"}

# Combine the main file and the Swiss/Nordic file into one set of priced bank names
ds = ds_names("banks_ri.xlsm") | ds_names("scandinavian_banks_ri.xlsm")

In [ ]:
# Sigwatch name -> Datastream name, parsed from the Stata file
mp = {}
for line in open(DATA / "example_name_cleaning.txt", encoding="utf-8", errors="ignore"):
    if line.strip().startswith("replace company"):
        q = re.findall(r'"([^"]*)"', line)                      # [target, source1, source2, ...]
        mp.update({s.strip().lower(): q[0].strip().lower() for s in q[1:]})

# remove wrong mappings from the Stata file (e.g. Berkshire Hathaway -> a Swiss cantonal bank)
for k in ["berkshire hathaway", "pinnacle west capital corporation", "virgin group",
          "lloyd's of london", "ace group", "gruppo assicurazioni generali"]:
    mp.pop(k, None)

# Banco Popular Espanol has no price series: Datastream's "POPULAR" is Popular Inc. of Puerto Rico, a different bank
mp = {k: v for k, v in mp.items() if v != "popular"}

# fix targets that don't exist in Datastream + add banks the Stata file missed
mp.update({
    "bankinter, s.a.": "bankinter 'r'", "allianz se": "allianz", "allianz group / allianz se": "allianz",
    "jp morgan chase": "jp morgan chase & co.", "goldman sachs group, inc.": "goldman sachs gp.",
    "skandinaviska enskilda banken ab (seb)": "skandinaviska enskilda banken a",
    "handelsbanken": "svenska handelsbanken a", "dnb nor": "dnb", "u.s. bank": "us bancorp", "la caixa": "caixabank",
})

In [ ]:
# Match on the company, else on its parent (so subsidiaries like Halifax -> Lloyds are kept)
for c in ["company", "company_parent"]:
    s = sig[c].str.strip().str.lower()
    sig[c + "_ds"] = s.map(mp).fillna(s)
sig["ds_name"] = sig["company_ds"].where(sig["company_ds"].isin(ds),
                                         sig["company_parent_ds"].where(sig["company_parent_ds"].isin(ds)))

# Saga Group's Datastream index never changes (no price data), so its events cannot be studied
NO_PRICES = {"saga group"}
matched = sig[sig["ds_name"].notna() & ~sig["ds_name"].isin(NO_PRICES)].copy()

# Datastream's bank lists also contain insurers, an investment trust and a card network: these are not banks
NON_BANKS = {"allianz", "aegon", "asr nederland", "admiral group", "aflac", "alliance trust", "mastercard"}

# Project scope: US, UK and EU banks, so Swiss and Norwegian banks are out
EU = {"Austria", "Belgium", "Bulgaria", "Croatia", "Cyprus", "Czech Republic", "Denmark", "Estonia", "Finland", "France",
      "Germany", "Greece", "Hungary", "Ireland", "Italy", "Latvia", "Lithuania", "Luxembourg", "Malta", "Netherlands",
      "Poland", "Portugal", "Romania", "Slovakia", "Slovenia", "Spain", "Sweden"}
SCOPE = EU | {"US", "UK"}

banks = matched[~matched["ds_name"].isin(NON_BANKS) & matched["parent_country"].isin(SCOPE)].copy()

out = matched[~matched.index.isin(banks.index)]
print(f"{matched['ds_name'].nunique()} priced firms matched -> {banks['ds_name'].nunique()} US/UK/EU banks")
print(out.groupby(["parent_country", "ds_name"], observed=True).size().rename("rows dropped").to_string())

## A3. Name → ISIN lookup
One ISIN per bank. Datastream's ISIN comes first because it belongs to the series we price, then Sigwatch's. Three banks have neither and are filled by hand.

In [ ]:
# Datastream's ISIN per bank, from the list sheets of banks_info.xlsx
sheets = pd.read_excel(DATA / "banks_data_bocconi" / "banks_info.xlsx", sheet_name=None)
info = pd.concat([v for k, v in sheets.items() if k.startswith("LBANKS") and "ISIN CODE" in v], ignore_index=True)
info["ds_name"] = info["NAME"].astype(str).str.strip().str.lower()
ds_isin = info.dropna(subset=["ISIN CODE"]).drop_duplicates("ds_name").set_index("ds_name")["ISIN CODE"]

# Sigwatch's most frequent ISIN per bank
sig_isin = banks.dropna(subset=["isin_1"]).groupby("ds_name")["isin_1"].agg(lambda s: s.value_counts().index[0])

# No ISIN in either source
MANUAL_ISIN = {"citigroup": "US1729674242", "ally financial": "US02005N1000", "skandinaviska enskilda banken a": "SE0000148884"}

lookup = banks[["ds_name", "company", "company_parent", "parent_country"]].drop_duplicates().sort_values(["ds_name", "company"])
lookup["isin"] = lookup["ds_name"].map(ds_isin).fillna(lookup["ds_name"].map(sig_isin)).fillna(lookup["ds_name"].map(MANUAL_ISIN))
lookup["isin_sigwatch"] = lookup["ds_name"].map(sig_isin)                 # can differ: an ISIN changes after a merger or reverse split

one = lookup.drop_duplicates("ds_name")
print(f"{len(one)} banks | {one['isin'].isna().sum()} without an ISIN | "
      f"{(one['isin_sigwatch'].notna() & (one['isin'] != one['isin_sigwatch'])).sum()} where Sigwatch's ISIN differs from Datastream's")

#### **Description of the bank campaigns**

In [ ]:
# Descriptives of the bank campaigns

# Sigwatch gives one reach score per NGO (up to 5): keep the strongest one in the event
banks["ngo_power"] = banks[[f"ngo_power{i}" for i in range(1, 6)]].max(axis=1)

# Size of the sample: rows, events, banks and lead NGOs
print(f"{len(banks):,} rows | {banks['uid_archive'].nunique():,} events | "
      f"{banks['ds_name'].nunique()} banks | {banks['ngo_name1'].nunique():,} lead NGOs")

# Positive vs. negative campaigns
print(banks["sentiment"].map({-2: "very negative", -1: "negative", 0: "neutral", 1: "positive", 2: "very positive"})
      .value_counts().reindex(["very negative", "negative", "neutral", "positive", "very positive"]))

# Distribution of prominence, NGO reach and number of firms named in the same event
print(banks[["prominence", "ngo_power", "n_targets"]].describe().round(2))

# Most active NGOs
print(banks["ngo_name1"].value_counts().head(10))

**Sentiment**: The sample contains 3,212 campaign-bank observations (1,311 distinct events) targeting 59 listed banks, launched by 362 different lead NGOs. Campaigns are overwhelmingly critical: about 81% are negative or very negative (2,596 rows), versus about 14% positive (442) and 5% neutral (174). This is consistent with NGOs mostly using campaigns to pressure banks rather than to praise them, so the event study will have far more negative than positive events.

**Prominence**: Prominence is centred on 2, so a typical campaign mentions the bank in the body of the communication rather than the headline. The lead NGO’s reach is medium, with a quarter of the sample at 2.5 or above (global or international NGOs). n_targets is skewed: the median event names 8 firms but the maximum is 72. Many bank campaigns are therefore rankings or multi-company reports rather than bank-specific attacks. We flag events with more than 10 targets (mass) and treat them as a robustness check.

**Activity**: Activity is concentrated in a few NGOs: Friends of the Earth France alone accounts for 240 rows (about 7.5% of the sample), followed by Which? (UK, 177) and Rainforest Action Network (US, 120). The top ten are mostly national or regional groups (France, UK, US, Belgium, Netherlands, Spain), so the results may reflect these NGOs’ repeated campaigns rather than the whole NGO universe.

#### **One row per bank-day, then flag problem events**

**One row per bank-day**
A bank can appear several times on the same day (parent and subsidiary, or several NGOs), so we collapse to one row per bank and date. We average sentiment, keep the maximum prominence and NGO reach, and add a `climate` flag from the issue fields.

In [ ]:
#One row per bank-day
issues = banks[["issue_name1", "issue_name2", "issue_name3"]].fillna("").agg(" ".join, axis=1)
banks["climate"] = issues.str.contains(r"climate|carbon|coal|fossil|(?<!palm )\boil|\bgas\b|pipeline|fracking|emission|renewable", case=False)

events = (banks.sort_values(["ds_name", "date"])
          .groupby(["ds_name", "date"], as_index=False)
          .agg(uid_archive=("uid_archive", "first"), uids=("uid_archive", lambda s: ";".join(map(str, s.unique()))),
               parent_country=("parent_country", "first"), ngo=("ngo_name1", "first"), n_rows=("uid_archive", "size"),
               sentiment=("sentiment", "mean"), min_sentiment=("sentiment", "min"),
               prominence=("prominence", "max"), ngo_power=("ngo_power", "max"),
               n_targets=("n_targets", "min"), climate=("climate", "max")))
events["negative"] = (events["sentiment"] < 0).astype(int)
events["isin"] = events["ds_name"].map(one.set_index("ds_name")["isin"])

**Flag problem events and select the final sample**
For the event study we need clean events. We (1) drop days where the same bank got opposite-sign news within 5 days, since we can't tell which news moved the price, and (2) keep only the first event in each 30-day run, so the [-10, +10] windows of kept events don't overlap. Events naming more than 10 firms (`mass`) are only flagged, not dropped.

The overlap rule stays here for now because Part C reads `events["keep"]`. `gap` is an argument, so other rules can be tested.

In [ ]:
# Flag problem events and keep a clean, non-overlapping sample
g = events.groupby("ds_name")
events["days_since_prev"] = g["date"].diff().dt.days

# Bank-days with opposite-sign news within 5 days (can't tell which moved the price)
events["opposite"] = (events["days_since_prev"] <= 5) & (events["sentiment"] * g["sentiment"].shift() < 0)
events["opposite"] |= events["opposite"].groupby(events["ds_name"]).shift(-1, fill_value=False)
events["mass"] = events["n_targets"] > 10                    # flag only, not dropped

# Keep an event only if it is >= gap days after the last kept one
def first_in_window(dates, gap=30):
    keep, last = [], None
    for d in dates:
        ok = last is None or (d - last).days >= gap
        keep.append(ok)
        last = d if ok else last
    return keep

# Apply it bank by bank, after removing the opposite-sign events
def overlap_rule(events, gap=30):
    ok = events[~events["opposite"]]
    keep = pd.Series(False, index=events.index)
    keep.loc[ok.index] = ok.groupby("ds_name")["date"].transform(lambda s: pd.Series(first_in_window(s, gap), index=s.index))
    return keep.astype(bool)


events["keep"] = overlap_rule(events)
print(events[["opposite", "mass", "keep", "climate"]].sum(), len(events))

Out of 2,935 bank-days, 231 (8%) have opposite-sign news within 5 days, 1,013 (35%) are flagged as mass events and 1,135 (39%) carry a climate tag. After removing the opposite-sign days, the 30-day rule leaves 1,363 events with non-overlapping [−10, +10] windows. We keep `mass` and `climate` as flags, so later we can test whether the effect differs for bank-specific vs. multi-firm campaigns, and for climate vs. other campaigns, before and after COP21.

In [ ]:
# Map each event to a trading day
def calendar(file):                                                      # trading days from the Fama-French file
    d = pd.read_excel(DATA / "banks_data_bocconi" / file, usecols=["date"])["date"]
    return pd.to_datetime(d, format="%m/%d/%Y").sort_values().to_frame("trade_date")

cal = {"US": calendar("North_America_3_Factors_Daily.xlsx"), "EU": calendar("Europe_3_Factors_Daily.xlsx")}
events["region"] = np.where(events["parent_country"] == "US", "US", "EU")     # US banks: US calendar, others: Europe

# event day = first trading day on or after the campaign date
events = pd.concat([pd.merge_asof(g.sort_values("date"), cal[r], left_on="date", right_on="trade_date", direction="forward")
                    for r, g in events.groupby("region")]).sort_values(["ds_name", "date"]).reset_index(drop=True)
events["shifted"] = events["trade_date"] != events["date"]
print(events["shifted"].sum(), "of", len(events), "events moved to the next trading day")

**Comment:** All 2,935 bank-days already fall on trading days (Monday to Friday, none on a holiday), so no event needed to be moved. The column `trade_date` is kept as the event day for the event study.

### **Figure 1: When, against whom, by whom**
We plot all bank-days with a campaign (before the 30-day rule) to see how the campaigns are spread over time, across banks and across NGOs.

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter
import matplotlib.dates as mdates

# Palette: red = negative, green = positive, grey = neutral (same as the PC Labs)
C = {"neg": "#c8283c", "neu": "#b7b5ad", "pos": "#1f8a5b", "navy": "#0d3b78", "grey": "#52514e"}
events["sign"] = np.select([events["sentiment"] < 0, events["sentiment"] > 0], ["neg", "pos"], "neu")
COP21 = pd.Timestamp("2015-12-12")                         # Paris Agreement

fig, axes = plt.subplots(1, 3, figsize=(18, 5.2), gridspec_kw={"width_ratios": [1.3, 1, 1.1], "wspace": 0.95})

# A. campaigns per quarter by tone
q = events.groupby([events["date"].dt.to_period("Q").dt.to_timestamp(), "sign"]).size().unstack(fill_value=0)
ax = axes[0]; bottom = np.zeros(len(q))
for s, lab in [("neg", "Negative"), ("neu", "Neutral"), ("pos", "Positive")]:
    ax.bar(q.index, q[s], width=70, bottom=bottom, color=C[s], label=lab); bottom += q[s].values
ax.axvline(COP21, color="black", ls=":", lw=1); ax.text(COP21, ax.get_ylim()[1] * 0.97, " COP21", va="top", fontsize=9)
ax.set_title("A. Campaigns per quarter, by tone", loc="left", fontweight="bold")
ax.set_ylabel("Bank-days with a campaign"); ax.legend(frameon=False, loc="upper left", fontsize=9)

# B. most targeted banks
top = events.groupby("ds_name").size().nlargest(12).index
b = events[events["ds_name"].isin(top)].groupby(["ds_name", "sign"]).size().unstack(fill_value=0).reindex(top)[::-1]
ax = axes[1]; left = np.zeros(len(b))
for s in ["neg", "neu", "pos"]:
    ax.barh(b.index.str.title(), b[s], left=left, color=C[s], height=0.7); left += b[s].values
ax.set_title("B. 12 most targeted banks", loc="left", fontweight="bold"); ax.set_xlabel("Bank-days with a campaign")

# C. most active NGOs
ngo_top = events.groupby("ngo").size().nlargest(10)[::-1]
ax = axes[2]; ax.barh([x[:26] + ("…" if len(x) > 26 else "") for x in ngo_top.index], ngo_top.values, color=C["navy"], height=0.7)
ax.set_title("C. 10 most active lead NGOs", loc="left", fontweight="bold"); ax.set_xlabel("Bank-days with a campaign")

for ax in axes: ax.spines[["top", "right"]].set_visible(False); ax.grid(alpha=0.25, axis="x" if ax is not axes[0] else "y"); ax.set_axisbelow(True)

# Panel A: one x label every 2 years (must come after the figure is created)
axes[0].xaxis.set_major_locator(mdates.YearLocator(2))
axes[0].xaxis.set_major_formatter(mdates.DateFormatter("%Y"))

share = (events["sign"] == "neg").mean()
fig.suptitle(f"Figure 1: Bank campaigns are mostly negative ({share:.0%}) and concentrated in a few banks and NGOs",
             x=0.01, ha="left", fontsize=13.5, fontweight="bold")
fig.text(0.01, -0.02, f"{len(events):,} bank-days, {events['ds_name'].nunique()} banks, 2010-2018. "
         "One row per bank and day (all sentiments, before the 30-day rule). COP21 = Paris Agreement, 12 Dec 2015.", fontsize=9, color=C["grey"])
plt.show()

**Comment:**

**A.** Campaigns against banks rise from about 20-80 per quarter in 2010–2011 to peaks of 120–155 in 2017–2018, and red (negative) dominates in every quarter. The positive share is largest in 2015 (about 23%), the year of COP21, when NGOs also praised banks for climate commitments.

**B.** A few large banks are the main targets: HSBC (214 bank-days), BNP Paribas (201) and Deutsche Bank (193). Deutsche Bank is almost purely negative, while BNP Paribas and ING also receive a visible share of positive campaigns.

**C.** Activity is concentrated in a handful of NGOs: Friends of the Earth France leads with 224 bank-days, followed by Which? (UK) at 131 and Rainforest Action Network at 109. Most are national or regional groups (France, UK, US, Belgium, Netherlands, Spain), so a few NGOs' repeated campaigns drive much of the sample.

### Figure 2: What a campaign looks like
We describe the three Sigwatch intensity variables: where the bank is mentioned (`prominence`), the reach of the strongest NGO (`ngo_power`) and how many firms are named in the same communication (`n_targets`).

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.6), gridspec_kw={"wspace": 0.3})

# A. prominence, negative vs positive campaigns (share within each tone)
ax = axes[0]; w = 0.38; lv = [1, 2, 3, 4]
for k, (s, lab) in enumerate([("neg", "Negative"), ("pos", "Positive")]):
    sh = events[events["sign"] == s]["prominence"].value_counts(normalize=True).reindex(lv, fill_value=0)
    ax.bar(np.array(lv) + (k - 0.5) * w, sh.values, width=w, color=C[s], label=lab)
ax.set_xticks(lv, ["1\nreport only", "2\nelsewhere", "3\nfirst paragraph", "4\nheadline"], fontsize=9)
ax.yaxis.set_major_formatter(PercentFormatter(1, decimals=0)); ax.legend(frameon=False)
ax.set_title("A. Where is the bank mentioned?", loc="left", fontweight="bold"); ax.set_ylabel("Share of campaigns")

# B. reach of the strongest NGO
ax = axes[1]; p = events["ngo_power"].value_counts(normalize=True).sort_index()
ax.bar(p.index.astype(str), p.values, color=C["navy"], width=0.7)
ax.yaxis.set_major_formatter(PercentFormatter(1, decimals=0)); ax.set_xlabel("ngo_power (0.5 = local, 2.75 = global)")
ax.set_title("B. Reach of the strongest NGO", loc="left", fontweight="bold")

# C. firms named in the same communication
ax = axes[2]; clip = events["n_targets"].clip(upper=30)
ax.hist(clip, bins=np.arange(0.5, 31.5, 1), color=C["navy"], weights=np.ones(len(clip)) / len(clip))
ax.axvline(10.5, color=C["neg"], ls="--", lw=1.2)
ax.text(11, ax.get_ylim()[1] * 0.92, f"{events['mass'].mean():.0%} of campaigns\nname more than 10 firms\n(flagged 'mass')", color=C["neg"], fontsize=9, va="top")
ax.yaxis.set_major_formatter(PercentFormatter(1, decimals=0)); ax.set_xlabel("Firms named in the same communication (30 = 30 or more)")
ax.set_title("C. Many campaigns are not bank-specific", loc="left", fontweight="bold")

for ax in axes: ax.spines[["top", "right"]].set_visible(False); ax.grid(alpha=0.25, axis="y"); ax.set_axisbelow(True)
fig.suptitle("Figure 2: A typical campaign is mid-prominence, from a mid-reach NGO, and often names many firms",
             x=0.01, ha="left", fontsize=13.5, fontweight="bold", y=1.03)
plt.show()

**Comment:**

**A.** Negative campaigns put the bank in more visible places than positive ones: about 24% mention it in the headline versus 15% of positive campaigns. Positive news is mostly mentioned elsewhere in the text (about 61%).

**B.** The reach of the lead NGO is bimodal: about 36% of campaigns come from NGOs with reach 1.0 (local or national groups) and 32% from NGOs at 2.5 (international groups). Very local (0.5) and very global (2.75) NGOs are rare.

**C.** Most campaigns name few firms (a peak at 5), but 35% name more than 10, and a small group names 30 or more. These are bank rankings and multi-company reports, not bank-specific news, so we flag them as `mass` and use them as a robustness check.

## A4. Handoff files
- `events_banks.csv`: one row per bank-day, with the flags and the ISIN. **C uses it for the event study.** `uids` links each row back to its campaigns.
- `campaigns_banks.csv`: one row per campaign, with the report text and links. **D uses it for the add-on, and A for the NLP step.**
- `lookup_banks.csv`: Sigwatch name → Datastream name → ISIN.

In [ ]:
# One row per campaign, with the text and links
campaigns = (banks.groupby("uid_archive", as_index=False)
             .agg(date=("date", "first"), ngo=("ngo_name1", "first"), ngo_power=("ngo_power", "max"),
                  sentiment=("sentiment", "mean"), prominence=("prominence", "max"), climate=("climate", "max"),
                  n_targets=("n_targets", "first"), n_banks=("ds_name", "nunique"),
                  banks=("ds_name", lambda s: "; ".join(sorted(set(s)))),
                  issue_name1=("issue_name1", "first"), issue_name2=("issue_name2", "first"), issue_name3=("issue_name3", "first"),
                  report=("report", "first"), link1=("link1", "first"), link2=("link2", "first"), link3=("link3", "first"),
                  link4=("link4", "first"), link5=("link5", "first")))
campaigns["report"] = campaigns["report"].fillna("").str.replace(r"<[^>]+>", " ", regex=True).map(html.unescape).str.strip()   # HTML -> plain text

events.to_csv(OUT / "events_banks.csv", index=False)
campaigns.to_csv(OUT / "campaigns_banks.csv", index=False)
lookup.to_csv(OUT / "lookup_banks.csv", index=False)
print(f"{len(events):,} bank-days | {len(campaigns):,} campaigns | {(campaigns['report'].str.len() > 0).mean():.0%} with report text, "
      f"{(campaigns['link1'].fillna('').str.len() > 0).mean():.0%} with a link")

## A5. NLP sentiment and link check (#3)

We score each campaign from the `report` text in `campaigns_banks.csv`, compare the score with Sigwatch's own tone, test whether the source links still work, and merge the results back to bank-days through `uids`.

**Output:** `handoff/campaigns_banks_nlp.csv` (one row per campaign) and `handoff/events_banks_nlp.csv` (same rows as `events_banks.csv`, with nlp_score, nlp_score_min, nlp_score_link, vader, any_link_ok and nlp_negative).


**Load the campaigns.** We read the file written in A4 (one row per campaign). `uid_archive` is read as text so that it matches the `uids` column of `events_banks.csv` later. We also check how long the texts are.

In [ ]:
from pathlib import Path
import re, html
import numpy as np
import pandas as pd

# Folder of the handoff files: reuse OUT from the top of the notebook if it exists
OUT = Path(OUT) if "OUT" in globals() else Path("handoff")

# Read the campaign file; uid_archive as text (it is joined with the `uids` text column at the end)
camp = pd.read_csv(OUT / "campaigns_banks.csv", dtype={"uid_archive": str}, parse_dates=["date"])
camp["report"] = camp["report"].fillna("").astype(str)         # an empty report becomes an empty string, not NaN

# Quick look: number of campaigns, share with text, and length of the texts in words
print(f"{len(camp):,} campaigns | {(camp['report'].str.len() > 0).mean():.0%} with text")
print(camp["report"].str.split().str.len().describe().round(0))

**Baseline score: VADER.** VADER is a simple general-purpose word list. It runs in seconds and is not built for finance, so we use it as a point of comparison for FinBERT, not as the main score.

In [ ]:
# pip install vaderSentiment
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
vader = SentimentIntensityAnalyzer()

def vader_score(text):
    # VADER saturates on long texts, so we score sentence by sentence and average
    sents = [s for s in re.split(r"(?<=[.!?])\s+", text) if len(s) > 3]
    if not sents:
        return np.nan
    return float(np.mean([vader.polarity_scores(s)["compound"] for s in sents]))   # in [-1, +1]

camp["vader"] = camp["report"].map(vader_score)
print(camp["vader"].describe().round(3))

**Main score: FinBERT.** 

FinBERT is a BERT model trained on financial text. For each text it gives the probabilities of positive, negative and neutral. Our score is `P(positive) - P(negative)`, between -1 (very negative) and +1 (very positive). BERT reads at most 512 tokens, so long reports are cut into chunks and the chunk probabilities are averaged. Scoring is slow (about 5-15 minutes on a CPU), so the results are saved to a file and a rerun only scores new campaigns.

In [ ]:
# pip install transformers torch   (the first run downloads about 440 MB)
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# --- Load the model ---
MODEL = "ProsusAI/finbert"
tok = AutoTokenizer.from_pretrained(MODEL)                      # cuts text into tokens
model = AutoModelForSequenceClassification.from_pretrained(MODEL).eval()   # eval(): prediction mode, no training
label_idx = {v.lower(): k for k, v in model.config.id2label.items()}       # position of each label, read from the model
MAX_TOK, MAX_CHUNKS = 510, 6                                    # 510 tokens + [CLS] + [SEP] = 512; at most 6 chunks per text

# --- Score one text ---
def finbert_probs(text):
    ids = tok(text, add_special_tokens=False, truncation=False)["input_ids"]       # the whole text as token ids
    if not ids:
        return (np.nan, np.nan, np.nan)                         # empty text -> no score
    chunks = [ids[i:i + MAX_TOK] for i in range(0, len(ids), MAX_TOK)][:MAX_CHUNKS]   # cut into chunks of 510 tokens
    batch = [[tok.cls_token_id] + c + [tok.sep_token_id] for c in chunks]          # add the start and end markers BERT expects
    width = max(len(b) for b in batch)
    input_ids = torch.tensor([b + [tok.pad_token_id] * (width - len(b)) for b in batch])   # pad to the same length
    mask = (input_ids != tok.pad_token_id).long()               # tells the model which tokens are real
    with torch.no_grad():                                       # no training, so no gradients needed
        p = torch.softmax(model(input_ids=input_ids, attention_mask=mask).logits, dim=-1).numpy()   # probabilities per chunk
    w = np.array([len(c) for c in chunks], dtype=float)         # longer chunks weigh more
    p = (p * w[:, None]).sum(axis=0) / w.sum()                  # weighted average over the chunks
    return (p[label_idx["positive"]], p[label_idx["negative"]], p[label_idx["neutral"]])

# --- Cache: do not score twice ---
cache_file = OUT / "finbert_scores.csv"
done = (pd.read_csv(cache_file, dtype={"uid_archive": str}) if cache_file.exists()
        else pd.DataFrame(columns=["uid_archive", "fb_pos", "fb_neg", "fb_neu"]))   # scores from a previous run, if any
todo = camp[~camp["uid_archive"].isin(done["uid_archive"])]                          # only the campaigns not yet scored
print(f"{len(done):,} already scored, {len(todo):,} to score")

# --- Score the new campaigns (progress printed every 100) ---
rows = []
for n, (uid, text) in enumerate(zip(todo["uid_archive"], todo["report"]), 1):
    rows.append((uid, *finbert_probs(text)))
    if n % 100 == 0:
        print(f"  {n}/{len(todo)}")
done = pd.concat([done, pd.DataFrame(rows, columns=done.columns)], ignore_index=True)
done.to_csv(cache_file, index=False)                            # save, so a rerun does not start from zero

# --- Attach the scores to the campaigns and build the final score ---
camp = camp.merge(done, on="uid_archive", how="left", validate="1:1")
camp["nlp_score"] = camp["fb_pos"] - camp["fb_neg"]             # P(positive) - P(negative), in [-1, +1]
print(camp[["fb_pos", "fb_neg", "fb_neu", "nlp_score"]].describe().round(3))

**Validation.** Sigwatch already codes the tone of each campaign by hand (-2 to +2), so we can check whether the text score agrees with it: rank correlation, ability to separate positive from negative campaigns (AUC), sign agreement, average score by tone group, and a figure. About 81% of campaigns are negative, so the sign agreement is compared with the "always negative" baseline.

In [ ]:
from scipy.stats import spearmanr, mannwhitneyu
import matplotlib.pyplot as plt

# --- Prepare: keep campaigns with both scores, and put Sigwatch's mean tone into 5 groups ---
v = camp.dropna(subset=["nlp_score", "sentiment"]).copy()
v["tone"] = pd.cut(v["sentiment"], [-2.01, -1.5, -0.5, 0.5, 1.5, 2.01],
                   labels=["very negative", "negative", "neutral", "positive", "very positive"])

# --- 1. Rank correlation between each text score and Sigwatch's tone ---
for col in ["nlp_score", "vader"]:
    d = v.dropna(subset=[col])                                  # VADER can be missing for very short texts
    r, p = spearmanr(d[col], d["sentiment"])
    print(f"Spearman {col} vs Sigwatch tone: {r:.2f} (p = {p:.3g})")

# --- 2. AUC: probability that a random positive campaign scores higher than a random negative one (0.5 = chance, 1 = perfect) ---
pos, neg = v[v["sentiment"] > 0], v[v["sentiment"] < 0]
for col in ["nlp_score", "vader"]:
    a, b = pos[col].dropna(), neg[col].dropna()                 # drop missing scores before the test
    u = mannwhitneyu(a, b).statistic
    print(f"AUC {col} (positive vs negative campaigns): {u / (len(a) * len(b)):.2f}  [{len(a)} pos, {len(b)} neg]")

# --- 3. Sign agreement, compared with a model that always says 'negative' ---
nz = v[v["sentiment"] != 0]
print(f"Sign agreement: {(np.sign(nz['nlp_score']) == np.sign(nz['sentiment'])).mean():.1%} "
      f"(always-negative baseline: {(nz['sentiment'] < 0).mean():.1%})")

# --- 4. Average score by Sigwatch tone group (should rise from 'very negative' to 'very positive') ---
print(v.groupby("tone", observed=True)["nlp_score"].agg(["count", "mean", "median"]).round(3))

# --- Figure: 3 panels ---
C = globals().get("C", {"neg": "#c8283c", "neu": "#b7b5ad", "pos": "#1f8a5b", "navy": "#0d3b78", "grey": "#52514e"})   # palette of Figure 1
order = ["very negative", "negative", "neutral", "positive", "very positive"]
cols = [C["neg"], C["neg"], C["neu"], C["pos"], C["pos"]]       # red = negative, grey = neutral, green = positive

fig, axes = plt.subplots(1, 3, figsize=(17, 4.8), gridspec_kw={"wspace": 0.35})

# A. Mean score by Sigwatch tone, with 95% confidence interval and group size
ax = axes[0]
g = v.groupby("tone", observed=True)["nlp_score"].agg(["mean", "std", "count"]).reindex(order)
ci = 1.96 * g["std"] / np.sqrt(g["count"])                      # 95% confidence interval of the mean
ax.bar(range(5), g["mean"], yerr=ci, color=cols, alpha=0.7, capsize=4)
ax.axhline(0, color=C["grey"], lw=0.8)
ax.set_xticks(range(5)); ax.set_xticklabels([f"{t.replace(' ', chr(10))}\n(n={int(g.loc[t, 'count'])})" for t in order], fontsize=8.5)   # 'very negative' on two lines
ax.set_ylabel("Mean FinBERT score"); ax.set_title("A. Average score by Sigwatch tone", loc="left", fontsize=11)

# B. Share of campaigns that FinBERT scores as negative, by Sigwatch tone
ax = axes[1]
share = v.assign(fb_neg=v["nlp_score"] < 0).groupby("tone", observed=True)["fb_neg"].mean().reindex(order)
ax.bar(range(5), 100 * share, color=cols, alpha=0.7)
for i, s in enumerate(share):
    ax.text(i, 100 * s + 1.5, f"{100 * s:.0f}%", ha="center", fontsize=9)     # label each bar
ax.set_xticks(range(5)); ax.set_xticklabels([t.replace(" ", "\n") for t in order], fontsize=8.5)
ax.set_ylim(0, 100); ax.set_ylabel("% of campaigns with score < 0")
ax.set_title("B. Share scored negative", loc="left", fontsize=11)

# C. ROC curve: how well does the score separate positive from negative campaigns?
def roc(pos_s, neg_s):
    thr = np.sort(np.unique(np.r_[pos_s, neg_s]))[::-1]         # every possible cut-off, from high to low
    tpr = [(pos_s >= t).mean() for t in thr]                     # share of positive campaigns above the cut-off
    fpr = [(neg_s >= t).mean() for t in thr]                     # share of negative campaigns above the cut-off
    return np.r_[0, fpr, 1], np.r_[0, tpr, 1]

ax = axes[2]
for col, label, colour in [("nlp_score", "FinBERT", C["navy"]), ("vader", "VADER", C["grey"])]:
    d = v.dropna(subset=[col])                                   # VADER can be missing for very short texts
    p_s, n_s = d.loc[d["sentiment"] > 0, col].to_numpy(), d.loc[d["sentiment"] < 0, col].to_numpy()
    x, y = roc(p_s, n_s)
    ax.plot(x, y, color=colour, lw=2, label=f"{label} (AUC {np.sum(np.diff(x) * (y[1:] + y[:-1]) / 2):.2f})")   # AUC = area under the curve
ax.plot([0, 1], [0, 1], color=C["neu"], ls="--", lw=1, label="Chance (AUC 0.50)")
ax.set_xlabel("Share of negative campaigns above the cut-off"); ax.set_ylabel("Share of positive campaigns above the cut-off")
ax.set_title("C. Positive vs negative campaigns", loc="left", fontsize=11); ax.legend(frameon=False, fontsize=9)

for ax in axes:                                                  # common style for the three panels
    ax.spines[["top", "right"]].set_visible(False); ax.grid(alpha=0.25)
    ax.set_axisbelow(True)
plt.show()

# --- Read the biggest disagreements by hand: they show where the model fails ---
dis = v[(v["sentiment"] <= -1) & (v["nlp_score"] > 0.3) | (v["sentiment"] >= 1) & (v["nlp_score"] < -0.3)]
print(f"{len(dis)} strong disagreements")
for _, r in dis.head(5).iterrows():
    print(f"\n[{r['ngo']}] Sigwatch {r['sentiment']:+.1f}, score {r['nlp_score']:+.2f}\n{r['report'][:400]}")

**Comment:** The FinBERT score points in the right direction but is weak. It is positively correlated with Sigwatch's tone (Spearman 0.10, p < 0.001; VADER 0.13), and the average score rises from −0.26 for very negative campaigns to −0.11 for very positive ones, but the confidence intervals of neighbouring groups overlap (panel A). It separates positive from negative campaigns only slightly better than chance (AUC 0.60 for both FinBERT and VADER), and it scores 58% of positive and 49% of very positive campaigns as negative (panel B). Sign agreement (68.7%) is below the always-negative baseline (86.5%), so the score is not usable as a sign classifier.

The disagreements show why. Sigwatch codes the bank's stance relative to the NGO's demand, while FinBERT reads the wording. A campaign that praises banks for ending coal financing contains words such as "coal mining" and "cease their investment" and scores negative (Rainforest Action Network: Sigwatch +1, score −0.74), whereas a criticism of a sustainability award reads as positive (Deutsche Bank award: Sigwatch −2, score +0.36). We therefore keep Sigwatch's tone as the main sentiment variable and use the NLP score only as a continuous complement and robustness check.

**Step 1: test the links.** 

Each campaign has up to five source links (`link1` to `link5`). We list all of them, remove duplicates and test each one with a web request (32 in parallel). Results are saved in `link_check.csv`, so a rerun does not start from zero. For a first test, set `LIMIT = 50`; then set it back to None for the full run.

In [ ]:
import time
import requests
from urllib.parse import urlparse
from concurrent.futures import ThreadPoolExecutor, as_completed

# --- List all links: one row per (campaign, link) ---
links = (camp[["uid_archive", "date"] + [f"link{i}" for i in range(1, 6)]]
         .melt(id_vars=["uid_archive", "date"], var_name="slot", value_name="url"))   # link1..link5 -> one column
links["url"] = links["url"].fillna("").astype(str).str.strip()
links = links[links["url"].str.len() > 0].copy()                                    # drop empty slots
links.loc[~links["url"].str.match(r"https?://", case=False), "url"] = "http://" + links["url"]   # add a scheme when missing
print(f"{len(links):,} links, {links['url'].nunique():,} distinct URLs, {links['uid_archive'].nunique():,} campaigns")

# --- Test one URL ---
HEADERS = {"User-Agent": "Mozilla/5.0 (academic project, link check)"}              # some sites refuse requests without a browser-like name

def check(url):
    # Try HEAD first (light: no page download); if the server refuses, try GET. Returns (status, final URL, error)
    for method in ("HEAD", "GET"):
        try:
            r = requests.request(method, url, headers=HEADERS, timeout=(4, 6), allow_redirects=True, stream=True)   # 4 s to connect, 6 s to answer
            code, final = r.status_code, r.url
            r.close()
            if code < 400:
                return code, final, ""                           # the page answers
        except requests.exceptions.RequestException as e:
            err = type(e).__name__                               # Timeout, ConnectionError (dead domain), SSLError ...
            if method == "GET":
                return None, "", err                             # both methods failed
    return code, final, ""                                       # answered, but with an error code (404, 403, ...)

# --- Cache: do not test twice ---
link_cache = OUT / "link_check.csv"
checked = (pd.read_csv(link_cache) if link_cache.exists()
           else pd.DataFrame(columns=["url", "status", "final_url", "error"]))      # results of a previous run, if any
todo_urls = [u for u in links["url"].unique() if u not in set(checked["url"])]       # only the URLs not yet tested
LIMIT = None                                                    # put e.g. 50 here for a first test run
todo_urls = todo_urls[:LIMIT]
print(f"{len(checked):,} already checked, {len(todo_urls):,} to check")

# --- Progress bar (tqdm); falls back to a printed line every 100 links if tqdm is not installed ---
# pip install tqdm
try:
    from tqdm.auto import tqdm
    bar = tqdm(total=len(todo_urls), desc="Links checked", unit="link")
except ImportError:
    bar = None
count = {"working": 0, "broken (404/410)": 0, "blocked (401/403/429)": 0, "no answer": 0, "other error": 0}   # live tally

def tally(code):
    # Put one result in a rough category, only to show it live in the progress bar
    if code is None or pd.isna(code):
        return "no answer"
    if code < 400:
        return "working"
    if code in (404, 410):
        return "broken (404/410)"
    if code in (401, 403, 429):
        return "blocked (401/403/429)"
    return "other error"

# --- Test all new URLs, 32 at a time, with a checkpoint every 100 links ---
results, start = [], time.time()
with ThreadPoolExecutor(max_workers=32) as ex:
    futures = {ex.submit(check, u): u for u in todo_urls}          # start all the tests
    for n, fut in enumerate(as_completed(futures), 1):             # handle each result as soon as it arrives
        res = fut.result()
        results.append((futures[fut], *res))
        count[tally(res[0])] += 1
        if bar is not None:
            bar.update(1)
            bar.set_postfix({"ok": count["working"], "broken": count["broken (404/410)"],
                             "blocked": count["blocked (401/403/429)"], "no answer": count["no answer"]})
        elif n % 100 == 0:
            print(f"  {n}/{len(futures)} checked ({time.time() - start:.0f}s) {count}")
        if n % 100 == 0 or n == len(futures):
            # checkpoint: save everything done so far, so an interruption keeps the work
            (pd.concat([checked, pd.DataFrame(results, columns=checked.columns)], ignore_index=True)
               .drop_duplicates("url", keep="last").to_csv(link_cache, index=False))
if bar is not None:
    bar.close()

# --- Final save ---
checked = pd.concat([checked, pd.DataFrame(results, columns=checked.columns)], ignore_index=True).drop_duplicates("url", keep="last")
checked.to_csv(link_cache, index=False)
print(f"Done in {time.time() - start:.0f}s: {count}")

**Comment:** The campaigns list 2,188 source links, of which 2,175 are distinct URLs, spread over 1,310 of the 1,311 campaigns (one campaign has no link). We tested each URL once with a web request (HEAD, then GET if the server refuses). A link counts as working if the server answers with a code below 400; the results are saved in `link_check.csv`, so the test does not have to be run again. The test was run on the day of the analysis, so it measures link rot today, not when the campaigns were published.

**Step 2: classify and summarise.** 

Each result becomes one of: ok, redirected to the home page (the page was probably removed), gone (404/410), blocked (403/429: the site refuses scripts, so we cannot tell), or error (dead domain, timeout). We count "blocked" as working, so the share of dead links is a lower bound. We then look at the share of working links by year and by domain, and build per-campaign variables.

In [ ]:
# --- Classify one result ---
def classify(row):
    s = row["status"]
    if pd.isna(s):
        return "error (dead domain, timeout, SSL)"               # no answer at all
    s = int(s)
    # a redirect to the site's home page means that the page was removed, whatever the final status code
    o, f = urlparse(str(row["url"])), urlparse(str(row["final_url"]))
    if f.path in ("", "/") and o.path not in ("", "/"):
        return "redirected to home page"
    if s < 400:
        return "ok"
    if s in (404, 410):
        return "gone (404/410)"                                  # page removed
    if s in (401, 403, 429, 999):
        return "blocked (cannot tell)"                           # bot protection or login: the page may well exist
    return "error (other 4xx/5xx)"

checked["link_status"] = checked.apply(classify, axis=1)
links = links.merge(checked[["url", "link_status"]], on="url", how="left")
links["link_ok"] = links["link_status"].isin(["ok", "blocked (cannot tell)"])   # 'blocked' counts as alive: we cannot prove it is dead

# --- Summaries: overall, by year, by domain ---
print(links["link_status"].value_counts(normalize=True).round(3))
print("\nShare of links that work, by year of the campaign:")
print(links.groupby(links["date"].dt.year)["link_ok"].mean().round(3))      # old links should work less often
links["domain"] = links["url"].map(lambda u: urlparse(u).netloc.replace("www.", ""))
print("\nMost frequent domains and their share of links that work:")
print(links.groupby("domain")["link_ok"].agg(["size", "mean"]).sort_values("size", ascending=False).head(10).round(2))

# --- One row per campaign: number of links, share that work, and at least one working link ---
per_camp = (links.groupby("uid_archive")
            .agg(n_links=("url", "size"), share_links_ok=("link_ok", "mean"), any_link_ok=("link_ok", "max"))
            .reset_index())
camp = camp.drop(columns=["n_links", "share_links_ok", "any_link_ok"], errors="ignore")   # remove the copies from a previous run
camp = camp.merge(per_camp, on="uid_archive", how="left", validate="1:1")
print(f"\n{camp['any_link_ok'].mean():.1%} of campaigns keep at least one working link")

**Spot-check.** We print three links from the suspicious domains, with the server's answer, to confirm that home-page redirects and refusals are classified correctly.

In [ ]:
# Spot-check: a few links of the suspicious domains, with what the server answered
pd.set_option("display.max_colwidth", 90)
for d in ["eerlijkebankwijzer.nl", "wdm.org.uk", "globalwitness.org"]:
    print("\n", d)
    print(links[links["domain"] == d]
          .merge(checked[["url", "status", "final_url", "error"]], on="url")
          .head(3)[["url", "link_status", "status", "final_url", "error"]].to_string(index=False))

**Comment:** Of 2,175 distinct URLs, 38% still work, 4% are blocked (we cannot tell and count them as working), 4% redirect to the home page of the site (the page was removed), 39% are gone (404/410) and 16% give an error or no answer. Link rot rises with age: 29-33% of links work for campaigns of 2010-2014, against 44% for 2015 and 50-59% for 2016-2018. It also depends on the site: banktrack.org and corporateeurope.org keep 94-97% of their links, while eerlijkebankwijzer.nl and wdm.org.uk (now Global Justice Now, which redirects to its home page) keep none, and Global Witness keeps 16%, which points to site redesigns. 50.5% of campaigns still have at least one working link, so the `report` text, not the link, is the reliable source for the content of a campaign.

**Limits:** "blocked" links (403/429) are counted as working because the site may refuse scripts and still have the page; slow sites (more than 6 seconds) are counted as errors; and a page that answers "ok" may still show different content from the original.

**Read the linked pages.** Sigwatch's `report` is a short summary (a few lines), while the link leads to the full article or press release, which carries much more text and tone. For each campaign we take the first link that works and extract the main text of the page (without menus or cookie banners). Pages that are PDFs or have almost no text are skipped. Pages are saved in `link_texts.csv`, so a rerun does not download them again. For a first test, set `LIMIT_PAGES = 30`; then set it back to `None`.

In [ ]:
# pip install trafilatura   (extracts the main text of a web page and removes menus, ads, cookie banners)
import trafilatura

# --- Which page to read for each campaign: the first link that works ("ok"), in the order link1 ... link5 ---
good = (links[links["link_status"] == "ok"]
        .sort_values(["uid_archive", "slot"])
        .drop_duplicates("uid_archive")[["uid_archive", "url"]])
print(f"{len(good):,} campaigns have at least one working link")

# --- Download one page and extract its text ---
def fetch_text(url):
    try:
        r = requests.get(url, headers=HEADERS, timeout=(4, 10), stream=True)
        ctype = r.headers.get("Content-Type", "").lower()
        if "html" not in ctype:                                   # PDFs, images ... are skipped
            r.close()
            return "", "not html (" + ctype.split(";")[0] + ")"
        raw = r.raw.read(2_000_000, decode_content=True)          # at most 2 MB per page
        r.close()
        return trafilatura.extract(raw, include_comments=False, include_tables=False) or "", ""   # main text only
    except requests.exceptions.RequestException as e:
        return "", type(e).__name__

# --- Cache: do not download twice ---
text_cache = OUT / "link_texts.csv"
got = (pd.read_csv(text_cache, dtype={"uid_archive": str}).fillna("") if text_cache.exists()
       else pd.DataFrame(columns=["uid_archive", "url", "text", "error"]))           # pages from a previous run, if any
todo_pages = good[~good["uid_archive"].isin(got["uid_archive"])]
LIMIT_PAGES = None                                                 # put e.g. 30 here for a first test run
todo_pages = todo_pages.iloc[:LIMIT_PAGES]
print(f"{len(got):,} already downloaded, {len(todo_pages):,} to download")

# --- Download all new pages, 16 at a time, with a progress bar and a checkpoint every 50 pages ---
try:
    from tqdm.auto import tqdm
    bar = tqdm(total=len(todo_pages), desc="Pages downloaded", unit="page")
except ImportError:
    bar = None
rows = []
with ThreadPoolExecutor(max_workers=16) as ex:
    futures = {ex.submit(fetch_text, u): (uid, u) for uid, u in zip(todo_pages["uid_archive"], todo_pages["url"])}
    for n, fut in enumerate(as_completed(futures), 1):
        uid, u = futures[fut]
        text, err = fut.result()
        rows.append((uid, u, text, err))
        if bar is not None:
            bar.update(1)
        if n % 50 == 0 or n == len(futures):
            pd.concat([got, pd.DataFrame(rows, columns=got.columns)], ignore_index=True).to_csv(text_cache, index=False)   # checkpoint
if bar is not None:
    bar.close()
got = pd.concat([got, pd.DataFrame(rows, columns=got.columns)], ignore_index=True).drop_duplicates("uid_archive", keep="last")
got.to_csv(text_cache, index=False)

got["n_words"] = got["text"].str.split().str.len().fillna(0)
print(f"{(got['n_words'] >= 80).sum():,} pages with at least 80 words of text (usable)")
print(got.loc[got["n_words"] < 80, "error"].replace("", "page too short or empty").value_counts().head(6))

**Comment:** 453 pages gave at least 80 words of text. The others failed because the page was empty or had too little text (110, usually pages built with JavaScript or cookie walls), was a PDF (51), or was an image or gave a connection error (a handful). So we read a page for 35% of the campaigns, mostly the more recent ones.

**Score the pages.** We keep the pages with at least 80 words (a cookie banner or an error page has almost no text) and score them with the same FinBERT function as for the reports: `P(positive) - P(negative)`. Scores are saved in `link_finbert_scores.csv`. The page score becomes a second variable, `nlp_score_link`, next to the report score `nlp_score`.

In [ ]:
# --- Keep usable pages only (a cookie banner or an error page has almost no text) ---
MIN_WORDS = 80
pages = got[got["n_words"] >= MIN_WORDS][["uid_archive", "text", "n_words"]]

# --- Cache: do not score twice (same logic as for the reports; finbert_probs comes from the FinBERT cell above) ---
lk_cache = OUT / "link_finbert_scores.csv"
lk_done = (pd.read_csv(lk_cache, dtype={"uid_archive": str}) if lk_cache.exists()
           else pd.DataFrame(columns=["uid_archive", "lk_pos", "lk_neg", "lk_neu"]))
lk_todo = pages[~pages["uid_archive"].isin(lk_done["uid_archive"])]
print(f"{len(lk_done):,} pages already scored, {len(lk_todo):,} to score")

rows = []
for n, (uid, text) in enumerate(zip(lk_todo["uid_archive"], lk_todo["text"]), 1):
    rows.append((uid, *finbert_probs(text)))                        # P(positive), P(negative), P(neutral) of the page
    if n % 50 == 0:
        print(f"  {n}/{len(lk_todo)}")
        pd.concat([lk_done, pd.DataFrame(rows, columns=lk_done.columns)], ignore_index=True).to_csv(lk_cache, index=False)   # checkpoint
lk_done = pd.concat([lk_done, pd.DataFrame(rows, columns=lk_done.columns)], ignore_index=True)
lk_done[["lk_pos", "lk_neg", "lk_neu"]] = lk_done[["lk_pos", "lk_neg", "lk_neu"]].astype(float)    # make sure the scores are numbers
lk_done.to_csv(lk_cache, index=False)

# --- Attach to the campaigns; the page score is P(positive) - P(negative), like the report score ---
camp = camp.drop(columns=["lk_pos", "lk_neg", "lk_neu", "nlp_score_link", "link_words"], errors="ignore")   # remove old copies on a rerun
camp = (camp.merge(lk_done, on="uid_archive", how="left", validate="1:1")
            .merge(pages[["uid_archive", "n_words"]].rename(columns={"n_words": "link_words"}), on="uid_archive", how="left", validate="1:1"))
camp["nlp_score_link"] = camp["lk_pos"] - camp["lk_neg"]
print(f"{camp['nlp_score_link'].notna().sum():,} campaigns with a page score "
      f"({camp['nlp_score_link'].notna().mean():.0%}); median page length {camp['link_words'].median():.0f} words "
      f"vs {camp['report'].str.split().str.len().median():.0f} for the report")

**Comment:** The pages are much longer than the reports (median 538 words against 61). FinBERT reads at most 6 chunks of 510 tokens per page, so very long pages are cut. 35% of campaigns now have a page score.

**Does the full article give a better score than the report?** We compare the two scores on the same campaigns (those with both), so that the comparison is fair: rank correlation with Sigwatch's tone, AUC for positive against negative campaigns, and ROC curves. If the page score is clearly better, the longer text carries more information about the campaign.

In [ ]:
from scipy.stats import spearmanr

# --- Same campaigns for both scores, so that the comparison is fair ---
both = camp.dropna(subset=["nlp_score", "nlp_score_link", "sentiment"]).copy()
both["tone"] = pd.cut(both["sentiment"], [-2.01, -1.5, -0.5, 0.5, 1.5, 2.01],
                      labels=["very negative", "negative", "neutral", "positive", "very positive"])
pn = both[both["sentiment"] != 0].copy()                           # positive and negative campaigns only (for the AUC)
y = (pn["sentiment"] > 0).to_numpy()                               # True = positive campaign
s_rep, s_lnk = pn["nlp_score"].to_numpy(), pn["nlp_score_link"].to_numpy()
print(f"{len(both):,} campaigns with both scores ({y.sum()} positive, {(~y).sum()} negative)")

# --- AUC (probability that a random positive campaign scores above a random negative one) ---
def auc(s, y):
    p, n = s[y], np.sort(s[~y])
    lo, hi = np.searchsorted(n, p, "left"), np.searchsorted(n, p, "right")      # ties count for 1/2
    return (lo.sum() + 0.5 * (hi - lo).sum()) / (len(p) * len(n))

# --- Bootstrap: redraw the campaigns 1,000 times to get a 95% interval for each AUC and for their difference ---
rng = np.random.default_rng(0)
boot = []
for _ in range(1000):
    b = rng.integers(0, len(pn), len(pn))
    if y[b].sum() == 0 or (~y[b]).sum() == 0:
        continue
    boot.append((auc(s_rep[b], y[b]), auc(s_lnk[b], y[b])))
boot = np.array(boot)
ci_rep, ci_lnk = np.percentile(boot[:, 0], [2.5, 97.5]), np.percentile(boot[:, 1], [2.5, 97.5])
ci_diff = np.percentile(boot[:, 1] - boot[:, 0], [2.5, 97.5])
a_rep, a_lnk = auc(s_rep, y), auc(s_lnk, y)
print(f"AUC report:      {a_rep:.2f}  [95% CI {ci_rep[0]:.2f}, {ci_rep[1]:.2f}]")
print(f"AUC linked page: {a_lnk:.2f}  [95% CI {ci_lnk[0]:.2f}, {ci_lnk[1]:.2f}]")
print(f"Difference (page - report): {a_lnk - a_rep:+.2f}  [95% CI {ci_diff[0]:+.2f}, {ci_diff[1]:+.2f}]  -> a CI that contains 0 means no clear difference")
print(f"Spearman with Sigwatch tone: report {spearmanr(both['nlp_score'], both['sentiment'])[0]:.2f}, "
      f"page {spearmanr(both['nlp_score_link'], both['sentiment'])[0]:.2f}; "
      f"correlation between the two scores {spearmanr(both['nlp_score'], both['nlp_score_link'])[0]:.2f}")

# --- Figure: 3 panels ---
C = globals().get("C", {"neg": "#c8283c", "neu": "#b7b5ad", "pos": "#1f8a5b", "navy": "#0d3b78", "grey": "#52514e"})
order = ["very negative", "negative", "neutral", "positive", "very positive"]
fig, axes = plt.subplots(1, 3, figsize=(17, 4.8), gridspec_kw={"wspace": 0.35})

# A. ROC curves, with the bootstrap interval of each AUC in the legend
def roc(s, y):
    thr = np.sort(np.unique(s))[::-1]                              # every cut-off, from high to low
    return (np.r_[0, [(s[~y] >= t).mean() for t in thr], 1],       # share of negative campaigns above the cut-off
            np.r_[0, [(s[y] >= t).mean() for t in thr], 1])        # share of positive campaigns above the cut-off
ax = axes[0]
for s, lab, a, ci, colour in [(s_rep, "Report", a_rep, ci_rep, C["grey"]), (s_lnk, "Linked page", a_lnk, ci_lnk, C["navy"])]:
    x_, y_ = roc(s, y)
    ax.plot(x_, y_, color=colour, lw=2, label=f"{lab}: AUC {a:.2f} [{ci[0]:.2f}, {ci[1]:.2f}]")
ax.plot([0, 1], [0, 1], color=C["neu"], ls="--", lw=1, label="Chance (0.50)")
ax.set_xlabel("Share of negative campaigns above the cut-off"); ax.set_ylabel("Share of positive campaigns above the cut-off")
ax.set_title(f"A. Separating positive from negative\n({y.sum()} positive, {(~y).sum()} negative)", loc="left", fontsize=11)
ax.legend(frameon=False, fontsize=8.5, loc="lower right")

# B. Mean score by Sigwatch tone, for both text sources, with 95% interval
ax = axes[1]
w = 0.38
for k, (col, lab, colour) in enumerate([("nlp_score", "Report", C["grey"]), ("nlp_score_link", "Linked page", C["navy"])]):
    g = both.groupby("tone", observed=True)[col].agg(["mean", "std", "count"]).reindex(order)
    ax.bar(np.arange(5) + (k - 0.5) * w, g["mean"], w, yerr=1.96 * g["std"] / np.sqrt(g["count"]),
           color=colour, alpha=0.75, capsize=3, label=lab)
n_by_tone = both["tone"].value_counts().reindex(order)
ax.set_xticks(range(5)); ax.set_xticklabels([f"{t.replace(' ', chr(10))}\n(n={n_by_tone[t]})" for t in order], fontsize=8.5)
ax.axhline(0, color=C["grey"], lw=0.8); ax.set_ylabel("Mean FinBERT score")
ax.set_title("B. Average score by Sigwatch tone", loc="left", fontsize=11); ax.legend(frameon=False, fontsize=9)

# C. Report score against page score, one point per campaign
ax = axes[2]
for lab, mask, colour, size in [("Neutral", both["sentiment"] == 0, C["neu"], 14),
                                ("Negative", both["sentiment"] < 0, C["neg"], 16),
                                ("Positive", both["sentiment"] > 0, C["pos"], 22)]:
    ax.scatter(both.loc[mask, "nlp_score"], both.loc[mask, "nlp_score_link"], s=size, color=colour, alpha=0.55, label=f"{lab} (Sigwatch)")
ax.axhline(0, color=C["grey"], lw=0.6); ax.axvline(0, color=C["grey"], lw=0.6)
ax.set_xlabel("Score of the report"); ax.set_ylabel("Score of the linked page")
ax.set_title(f"C. The two scores (Spearman {spearmanr(both['nlp_score'], both['nlp_score_link'])[0]:.2f})", loc="left", fontsize=11)
ax.legend(frameon=False, fontsize=8.5)

for ax in axes:
    ax.spines[["top", "right"]].set_visible(False); ax.grid(alpha=0.25); ax.set_axisbelow(True)
plt.show()

**Comment:** On the 453 campaigns with both scores, the linked page does not do better than the report. 

- Panel A: the AUC for positive against negative campaigns is 0.54 [0.45, 0.63] for the page, an interval that contains chance (0.50), against 0.61 [0.52, 0.69] for the report. The difference (page − report) is −0.06 with a 95% CI of −0.18 to +0.05, which contains 0, so the gap is small. The rank correlation of the page score with Sigwatch's tone is even slightly negative (−0.05, against 0.10 for the report). 
- Panel B: the report score rises from −0.28 for very negative campaigns to −0.09 for very positive ones, while the page score shows no clear pattern (between −0.13 and +0.06). 
- Panel C: the two scores are unrelated (Spearman 0.06): most page scores sit close to 0, as if FinBERT read long articles as neutral, while many reports are scored very negative. A likely reason is that a long article mixes many topics, so the part about the bank is diluted when the whole text is averaged, whereas the report is a short summary of what the NGO claimed. The sample is small and selected: only 50 campaigns have a positive average tone (the AUC uses this split; the five rounded groups of panel B show 41 positive and very positive), and only campaigns whose link still works are included, mostly recent ones. The comparison therefore holds for these campaigns only. With more time we would score only the sentences that mention the bank, or ask a model whether the text criticises or praises the bank.

**Merge back to bank-days and save.** A bank-day in `events_banks.csv` can rest on several campaigns, listed in `uids` separated by `;`. We split `uids` into one row per (bank-day, campaign), attach the campaign scores, and average them back to one row per bank-day (the mean score, and the most negative one). The result has exactly the same rows as `events_banks.csv`, so C and Alissa can join it on `ds_name` and `date`. We save new files and do not overwrite the originals.

In [ ]:
# --- Read the bank-day file written in A4 (uids stays as text) ---
ev = pd.read_csv(OUT / "events_banks.csv", parse_dates=["date"], dtype={"uids": str})

# --- One row per (bank-day, campaign): split the ';'-separated uids and give each campaign its own row ---
m = (ev[["ds_name", "date", "uids"]]
     .assign(uid_archive=ev["uids"].str.split(";"))
     .explode("uid_archive"))
m = m.merge(camp[["uid_archive", "nlp_score", "nlp_score_link", "vader", "any_link_ok"]],
            on="uid_archive", how="left", validate="m:1")        # attach the campaign scores

# --- Back to one row per bank-day ---
agg = (m.groupby(["ds_name", "date"], as_index=False)
         .agg(nlp_score=("nlp_score", "mean"),                   # average over the campaigns of that bank-day
              nlp_score_min=("nlp_score", "min"),                # the most negative campaign of the day
              nlp_score_link=("nlp_score_link", "mean"),          # score of the linked page (only where a page was read)
              vader=("vader", "mean"),
              any_link_ok=("any_link_ok", "max")))               # at least one campaign with a working link
ev_nlp = ev.merge(agg, on=["ds_name", "date"], how="left", validate="1:1")
assert len(ev_nlp) == len(ev), "the merge must not add or drop bank-days"      # safety check
print(f"{ev_nlp['nlp_score'].notna().mean():.1%} of bank-days have an NLP score, "
      f"{ev_nlp['nlp_score_link'].notna().mean():.1%} have a linked-page score")

# --- A flag for the splits in Task 6: negative text (score < 0) or not ---
ev_nlp["nlp_negative"] = (ev_nlp["nlp_score"] < 0).astype(int)

# --- Save the two output files ---
camp.to_csv(OUT / "campaigns_banks_nlp.csv", index=False)        # per campaign: text, scores, links
ev_nlp.to_csv(OUT / "events_banks_nlp.csv", index=False)         # per bank-day: same rows as events_banks.csv, plus the NLP columns
print("saved:", OUT / "campaigns_banks_nlp.csv", "|", OUT / "events_banks_nlp.csv")

**Conclusion (NLP):** FinBERT scores about 70% of campaigns as negative. The score agrees only weakly with Sigwatch's tone: Spearman 0.10, AUC 0.60 for positive against negative campaigns (VADER: 0.13 and 0.60). The average score rises from −0.26 for very negative campaigns to −0.11 for very positive ones, but the sign agrees with Sigwatch in only 68.7% of campaigns, below an "always negative" rule (86.5%). Reading the full linked article does not help: the AUC of the page score is 0.54 against 0.61 for the report, and the difference (−0.06, 95% CI −0.18 to +0.05) is not clearly different from 0. We therefore keep Sigwatch's tone as the main sentiment variable, use the report score `nlp_score` only as a continuous complement and robustness check, and save the page score as `nlp_score_link`. 42% of the links still work (30% for campaigns in 2010-2012, 50-59% for 2016-2018) and 50.5% of campaigns keep at least one working link, so the `report` text, not the link, is the reliable source. **Limits:** FinBERT is trained on market news, and NGO reports criticise a bank rather than report bad news for its stock; mass events (more than 10 firms) describe many banks at once, so their text is not bank-specific; "blocked" links are counted as working, so the share of dead links is a lower bound.

---
# Part B: Market data and betas (owner: B, Cecilia)
Build the daily panel (returns from RI, Fama-French factors, risk-free rate, currencies), then estimate rolling CAPM and Fama-French betas (task #4), which give each bank's expected return on each day.

**Handoff to C:** `handoff/panel_returns.csv`, `handoff/betas_daily.csv`.

## B1. Daily panel
We build one daily panel with returns for the 59 banks in our events, plus the Fama-French factors and the risk-free rate. US banks use the North America factors and all other banks the Europe factors. We keep only banks that appear in the campaign data.

In [ ]:
# Total-return indices of all banks (main file + Swiss/Nordic file), same bank key as the events file
def load_ri(file):
    sheets = pd.read_excel(DATA / "banks_data_bocconi" / file, sheet_name=None)
    sheets.pop("REQUEST_TABLE")
    return pd.concat([v.rename(columns={"Name": "date"}).melt(id_vars="date", var_name="bank", value_name="ri").assign(country=k)
                      for k, v in sheets.items()], ignore_index=True)

ri = pd.concat([load_ri("banks_ri.xlsm"), load_ri("scandinavian_banks_ri.xlsm")], ignore_index=True)
ri["date"] = pd.to_datetime(ri["date"], errors="coerce")
ri["ds_name"] = ri["bank"].str.rsplit(" - ", n=1).str[0].str.strip().str.lower()
ri = ri[ri["ds_name"].isin(events["ds_name"])].dropna(subset=["date", "ri"]).sort_values(["ds_name", "date"]).drop_duplicates(["ds_name", "date"])

In [ ]:
# Daily returns; drop impossible moves (> +/-100% in a day = data errors) and flag very large ones (> 50%)
ri["ret"] = ri.groupby("ds_name")["ri"].pct_change()
ri = ri.dropna(subset=["ret"])
ri["ret"] = ri["ret"].where(ri["ret"].abs() <= 1)
ri["big"] = ri["ret"].abs() > 0.5

**Risk-free rate:** the `RF` column of the factor files is rounded to 0.01% a day, which makes it a coarse measure of the daily risk-free rate. We use the daily 4-week T-bill from `risk_free_rate.xlsm`, the closest match to the 1-month bill behind the Fama-French factors.

In [ ]:
# Daily 4-week US T-bill, annualised % -> daily decimal (actual/360)
rf = pd.read_excel(DATA / "banks_data_bocconi" / "risk_free_rate.xlsm", sheet_name="DAILY", header=2)
rf.columns = ["date", "tb3m", "tb4w"]
rf["date"] = pd.to_datetime(rf["date"], errors="coerce")
rf = rf.dropna(subset=["date"]).assign(rf=lambda d: pd.to_numeric(d["tb4w"]) / 100 / 360)[["date", "rf"]]

**Factors:** the data pack only has 3-factor files, so we download the daily 5-factor files (Mkt-RF, SMB, HML, RMW, CMA) for Developed Markets (Europe and North America) from Ken French's Data Library and save them as `North_America_5_Factors_Daily.xlsx` and `Europe_5_Factors_Daily.xlsx` in `banks_data_bocconi/`. `ff` below is therefore the Fama-French five-factor model (FF5), and the panel, the betas and C's `model="ff"` use all five factors.

In [ ]:
import io, zipfile, urllib.request

BASE = "https://mba.tuck.dartmouth.edu/pages/faculty/ken.french/ftp/"
FILES = {"Europe": "Europe_5_Factors_Daily_CSV.zip",
         "North_America": "North_America_5_Factors_Daily_CSV.zip"}

for name, zname in FILES.items():
    target = DATA / "banks_data_bocconi" / f"{name}_5_Factors_Daily.xlsx"
    if target.exists():
        continue
    with urllib.request.urlopen(BASE + zname, timeout=60) as r:
        z = zipfile.ZipFile(io.BytesIO(r.read()))
    lines = z.read(z.namelist()[0]).decode("latin-1").splitlines()
    h = next(i for i, l in enumerate(lines) if l.replace(" ", "").startswith(",Mkt-RF"))
    f = pd.read_csv(io.StringIO("\n".join(lines[h:])), index_col=0)
    f.index = f.index.astype(str).str.strip()
    f = f.loc[f.index.str.fullmatch(r"\d{8}")]
    f.columns = f.columns.str.strip()
    f = f.astype(float).replace(-99.99, np.nan)
    f.insert(0, "date", pd.to_datetime(f.index, format="%Y%m%d").strftime("%m/%d/%Y"))
    f.to_excel(target, index=False)
    print(name, f.shape, f["date"].iloc[0], "->", f["date"].iloc[-1], "|", list(f.columns))

In [ ]:
# Fama-French factors: % -> decimals, US banks use North America, all others Europe. The 5-factor file is used once it is in the folder
def load_ff(name, region):
    files = [DATA / "banks_data_bocconi" / f"{name}_{n}_Factors_Daily.xlsx" for n in (5, 3)]
    ff = pd.read_excel(next(f for f in files if f.exists()))
    ff["date"] = pd.to_datetime(ff["date"], format="%m/%d/%Y")
    ff.columns = ff.columns.str.lower().str.replace("-", "_")
    factors = ff.columns.drop(["date", "rf"])
    ff[factors] /= 100
    return ff.drop(columns="rf").assign(region=region)

ff = pd.concat([load_ff("North_America", "US"), load_ff("Europe", "EU")], ignore_index=True)
FACTORS = list(ff.columns.drop(["date", "region"]))
ri["region"] = np.where(ri["country"] == "US", "US", "EU")
print("Factors:", FACTORS)

**Currencies:** the Fama-French factors are in USD, but the bank series are in local currency: the Datastream request is plain `RI`, with no currency conversion. The `RF` column is a US rate in both factor files, so the European factors are USD returns over a US rate. The data pack has no European risk-free rate or factors in euro, so we convert bank returns to USD, which makes the US T-bill the right risk-free rate for every bank. This is the slides' warning that the risk-free rate differs across markets.

**Exchange rates:** `banks_data_bocconi/fx_usd.csv` is built from the ECB's daily euro reference rates (series `EXR.D.USD.EUR.SP00.A`, `EXR.D.GBP.EUR.SP00.A`, `EXR.D.SEK.EUR.SP00.A`), not part of the data pack. GBP and SEK are crossed through the euro. On days with no ECB rate, the last rate is carried forward.

**Limitation:** the ECB fixes its rates at 14:15 CET, before European markets close, so the FX move and the stock move of a day are not perfectly aligned.

In [ ]:
# Align currencies: bank returns -> USD, using fx_usd.csv (columns: date, EUR, GBP, SEK; USD per 1 unit of local currency)
CURRENCY = {"US": "USD", "UK": "GBP", "SD": "SEK"}                        # every other sheet in the sample is euro area
ri["currency"] = ri["country"].map(CURRENCY).fillna("EUR")
ri["ret_local"] = ri["ret"]                                               # kept for the stale-price check: FX moves would hide a zero return

fx_file = DATA / "banks_data_bocconi" / "fx_usd.csv"
USD_ALIGNED = fx_file.exists()
if USD_ALIGNED:
    fx = pd.read_csv(fx_file, parse_dates=["date"]).set_index("date").sort_index()
    fx = fx.reindex(pd.bdate_range(fx.index.min(), fx.index.max())).ffill().pct_change()       # FX holidays: rate carried forward
    fx = fx.rename_axis("date").reset_index().melt(id_vars="date", var_name="currency", value_name="fx_ret")
    ri = ri.merge(fx, on=["date", "currency"], how="left")
    ri["ret"] = (1 + ri["ret"]) * (1 + ri["fx_ret"].where(ri["currency"] != "USD", 0)) - 1
    print("Bank returns converted to USD")
else:
    print(f"WARNING: no {fx_file.name}. {ri.loc[ri['currency'] != 'USD', 'ds_name'].nunique()} non-US banks stay in local currency against USD factors")

In [ ]:
# Final panel: one row per bank and day, with excess return (return - risk-free) and a trading-day counter t
panel = ri.merge(ff, on=["date", "region"], how="inner").merge(rf, on="date", how="left")
assert panel["rf"].notna().all()
panel["exret"] = panel["ret"] - panel["rf"]
panel["mkt"] = panel["mkt_rf"] + panel["rf"]                              # market return
panel = panel[["date", "ds_name", "country", "currency", "region", "ri", "ret_local", "ret", "exret", *FACTORS, "mkt", "rf", "big"]].dropna(subset=["ret"])
panel = panel.sort_values(["ds_name", "date"]).reset_index(drop=True)
panel["t"] = panel.groupby("ds_name").cumcount()

# The region must agree between the events (from the parent's country) and the panel (from the Datastream sheet)
reg = events.drop_duplicates("ds_name").set_index("ds_name")["region"]
assert (reg == panel.groupby("ds_name")["region"].first().reindex(reg.index)).all()

panel.to_csv(OUT / "panel_returns.csv", index=False)
print(f"{len(panel):,} bank-days | {panel['ds_name'].nunique()} banks | {panel['date'].min().date()} to {panel['date'].max().date()}")
print(panel.groupby("region")["ds_name"].nunique())
print(panel.drop_duplicates("ds_name")["currency"].value_counts())

**Comment:** The panel has 172,502 daily observations for 59 banks (21 US, 38 European: 28 in euro, 7 in sterling, 3 in Swedish krona) from 2 January 2008 to 19 November 2020. The panel holds the total-return index `ri`, the bank return `ret`, the market return `mkt` and the risk-free rate `rf`, plus the factors. `ret` is in USD, while `ri` and `ret_local` stay in the original currency. Every bank in the campaign data has market data, so no bank is lost at this stage. Some events fall outside a bank's price history or lack a full estimation window, and C handles them in C2.

### Figures 3-4: How the bank panel behaves
We build an equal-weight portfolio of the banks in each region and compare it with the market. Figure 3 shows performance and Figure 4 shows how risky the banks are (beta) and how reliable their prices are.

In [ ]:
# Equal-weight bank portfolio, market, bank/market ratio and rolling 250-day beta, per region
plot = panel[panel["date"].between("2008-01-03", "2020-04-28")]           # the main price file stops on 28 Apr 2020
REG = {"US": ("US banks", C["navy"]), "EU": ("European banks", C["neg"])}
EVENTS = {"Lehman": "2008-09-15", "Euro crisis": "2011-08-01", "COP21": "2015-12-12", "Covid": "2020-03-11"}
res = {}
for r in REG:
    d = plot[plot["region"] == r]
    bank = d.groupby("date")["ret"].mean()
    mkt = d.groupby("date")[["mkt_rf", "rf"]].first().sum(axis=1)                    # market return = Mkt-RF + RF
    res[r] = pd.DataFrame({"bank": 100 * (1 + bank).cumprod(), "mkt": 100 * (1 + mkt).cumprod(),
                           "beta": bank.rolling(250).cov(mkt) / mkt.rolling(250).var()})
    res[r]["rel"] = 100 * res[r]["bank"] / res[r]["mkt"]

import matplotlib.ticker
def marks(ax):                                                            # vertical lines for the key episodes
    for lab, d in EVENTS.items():
        left = lab in ("Lehman", "Euro crisis")
        ax.axvline(pd.Timestamp(d), color="black", ls=":", lw=0.9)
        ax.text(pd.Timestamp(d), 1.0, (" " + lab) if left else (lab + " "), transform=ax.get_xaxis_transform(),
                fontsize=8.5, va="top", ha="left" if left else "right")
def style(axes):
    for ax in axes: ax.spines[["top", "right"]].set_visible(False); ax.grid(alpha=0.25); ax.set_axisbelow(True)

In [ ]:
# Figure 3 (performance)
fig, axes = plt.subplots(1, 2, figsize=(15, 5), gridspec_kw={"wspace": 0.3})

# A. levels, with end values labelled
ax = axes[0]
for r, (lab, col) in REG.items():
    ax.plot(res[r].index, res[r]["bank"], color=col, lw=1.6)
    ax.plot(res[r].index, res[r]["mkt"], color=col, lw=1.1, ls="--", alpha=0.55)
    ax.annotate(f"{lab}\n{res[r]['bank'].iloc[-1]:.0f}", (res[r].index[-1], res[r]["bank"].iloc[-1]), xytext=(5, 0), textcoords="offset points", fontsize=8.5, color=col, va="center")
    ax.annotate(f"market {res[r]['mkt'].iloc[-1]:.0f}", (res[r].index[-1], res[r]["mkt"].iloc[-1]), xytext=(5, 0), textcoords="offset points", fontsize=8, color=C["grey"], va="center")
ax.set_yscale("log"); ax.yaxis.set_minor_formatter(matplotlib.ticker.NullFormatter())
ax.set_yticks([30, 50, 100, 200]); ax.set_yticklabels(["30", "50", "100", "200"]); ax.set_xlim(right=pd.Timestamp("2022-01-01"))
marks(ax); ax.set_title("A. Banks (solid) vs market (dashed), 100 in Jan 2008", loc="left", fontweight="bold")

# B. bank / market ratio
ax = axes[1]
for r, (lab, col) in REG.items(): ax.plot(res[r].index, res[r]["rel"], color=col, lw=1.6, label=lab)
ax.axhline(100, color=C["grey"], lw=0.8); marks(ax)
ax.set_title("B. Banks relative to their market (100 = equal)", loc="left", fontweight="bold"); ax.legend(frameon=False, loc="lower center")

style(axes)
fig.suptitle("Figure 3: Banks fell harder than the market in crises, and European banks lag their market since 2010",
             x=0.01, ha="left", fontsize=13.5, fontweight="bold")
n_end = (panel.groupby("ds_name")["date"].max() <= "2020-04-28").sum()
cov = plot.groupby(["date", "region"])["ds_name"].nunique().unstack()
fig.text(0.01, -0.02, f"Equal-weight portfolio of the {panel['ds_name'].nunique()} event banks; market = Mkt-RF + RF. Plot stops on 28 Apr 2020, when {n_end} of {panel['ds_name'].nunique()} banks end. "
         f"Banks with data: {cov['EU'].min()}-{cov['EU'].max()} European, {cov['US'].min()}-{cov['US'].max()} US.", fontsize=8.5, color=C["grey"])
plt.show()

**Comment:** All lines are in USD.

**A.** Bank stocks fall much further than the market in every crisis. In 2008-09 the US bank portfolio loses about 70% and the European one about 80%, and both fall sharply again in the euro crisis (2011-12) and in the Covid crash (March 2020). Between the crises the two regions diverge. US banks stay above their market for most of 2017-2019, but after the Covid crash they stand at 176 against 233 for the market on 28 April 2020. European banks never get back to 100, staying around or below 80 after 2010, while their market rises to about 135 before Covid, and they end at 30 against 104. The COP21 line falls in a flat period for both, with no visible reaction at aggregate level.

**B.** The ratio makes the gap clear. US banks move between about 80 and 110 from 2011 until 2016, jump to about 120 after the US election at the end of 2016 (which matters for the Trump effect we test later), peak near 125 in 2018 and fall to about 75 in April 2020. European banks peak near 115 in 2009, fall to about 50 in 2012, recover to about 70 in 2014, and then drift down to about 38 before Covid and 27 in April 2020. The underperformance of European banks is persistent, not a one-off shock.

In [ ]:
#Figure 4 (risk and data quality)
# Share of days with exactly zero return per bank (stale prices), from 2010
zero = (panel[panel["date"] >= "2010-01-01"].assign(z=lambda d: d["ret_local"].abs() < 1e-12)
        .groupby("ds_name")["z"].mean().sort_values(ascending=False))
reg_of = panel.groupby("ds_name")["region"].first()
top = zero.head(12)[::-1]

fig, axes = plt.subplots(1, 2, figsize=(15, 5), gridspec_kw={"wspace": 0.55, "width_ratios": [1.2, 1]})

# A. rolling beta of the bank portfolio
ax = axes[0]
for r, (lab, col) in REG.items(): ax.plot(res[r].index, res[r]["beta"], color=col, lw=1.6, label=lab)
ax.axhline(1, color=C["grey"], lw=0.8); marks(ax)
ax.set_title("A. Bank portfolio beta (rolling 250 days)", loc="left", fontweight="bold")
ax.legend(frameon=False, loc="upper center", bbox_to_anchor=(0.55, 0.92))

# B. banks with the most stale prices
ax = axes[1]
ax.barh([s[:26] for s in top.index], top.values * 100, color=[REG[reg_of[s]][1] for s in top.index], height=0.65)
ax.set_title("B. Banks with the most zero-return days (2010-20)", loc="left", fontweight="bold")
ax.set_xlabel("% of trading days with zero return")

style(axes)
fig.suptitle("Figure 4: Bank betas move a lot over time, and a few banks have stale prices", x=0.01, ha="left", fontsize=13.5, fontweight="bold")
fig.text(0.01, -0.02, "Beta of the equal-weight bank portfolio on the market. Bars: red = European, blue = US bank. A zero return means the price did not change that day.",
         fontsize=8.5, color=C["grey"])
plt.show()

**Comment:**

**A.** The beta of the bank portfolio is far from constant. For US banks it ranges from about 0.9 to about 2.5: it peaks at about 2.5 at the start of 2010 after the financial crisis, falls to about 1.3 in 2011, rises again to about 1.7 in 2017, and falls back to about 0.9 in 2019 before rising with Covid. European banks mostly move between 1.0 and 1.6, peak at about 1.7 at the end of 2009 and reach a minimum of about 0.76 in 2018. They are close to US banks for much of the period, and above them in 2013-2015. A beta estimated in one period is therefore a poor guide to another, so each event must get its own beta from a window just before it (the 250 days ending 50 days before the event), not one beta for the whole sample.

**B.** A few banks have prices that do not change on many days (measured on local-currency returns, because FX moves would hide a zero return). Monte dei Paschi has a zero return on about 34% of days and Permanent TSB on about 16%. Most banks are far lower. Ten of the twelve banks shown are European (the exceptions are Ally Financial and Synovus). For these banks the beta is estimated with more noise, and an event day with no price change can hide a real reaction. We should check that these banks do not drive the results. We also dropped Saga Group, whose Datastream index never changes (100% zero returns), because it has no usable price data, and removed a duplicated Crédit Agricole series that created false zero returns.

## B2. Rolling betas and expected returns (#4)
For each bank and each day `t`, we regress the bank's excess return on the factors over the window [t−300, t−50], which ends 50 days before `t` as in Krüger (2015), so the estimate never uses the day it is applied to.

`betas_daily.csv` has one row per bank-day, with the parameters (`capm_alpha`, `capm_mkt_rf`, `ff_alpha`, `ff_mkt_rf`, `ff_smb`, `ff_hml`, `ff_rmw`, `ff_cma`) and the expected excess return for that day (`exp_capm`, `exp_ff`). The abnormal return of a day is `exret - exp_capm`.

**For C:** in the event study, read the parameters at the event day `t0` and hold them fixed over the whole event window, as described in the Part C notes.

In [ ]:
import statsmodels.api as sm
from statsmodels.regression.rolling import RollingOLS

EST_START, EST_END = 300, 50                                              # estimation window [t-300, t-50]

def rolling_params(d, factors, name):
    window = EST_START - EST_END + 1
    fit = RollingOLS(d["exret"], sm.add_constant(d[factors]), window=window).fit(params_only=True)
    p = fit.params.shift(EST_END)                                         # window ends EST_END days before t
    expected = p["const"] + (p[factors] * d[factors]).sum(axis=1, min_count=len(factors))
    return p.rename(columns=lambda c: f"{name}_alpha" if c == "const" else f"{name}_{c}").assign(**{f"exp_{name}": expected})

parts = []
for _, d in panel.groupby("ds_name"):
    parts.append(pd.concat([d[["date", "ds_name", "region", "t", "exret"]],
                            rolling_params(d, ["mkt_rf"], "capm"), rolling_params(d, FACTORS, "ff")], axis=1))
betas = pd.concat(parts, ignore_index=True).dropna(subset=["capm_alpha"])

betas.to_csv(OUT / "betas_daily.csv", index=False)
print(f"{len(betas):,} bank-days with betas | {betas['ds_name'].nunique()} banks")
print(betas.groupby("region")[["capm_mkt_rf", "ff_mkt_rf", "ff_smb", "ff_hml", "ff_rmw", "ff_cma"]].median().round(2))
print(f"Abnormal return, daily std: CAPM {(betas['exret'] - betas['exp_capm']).std():.4f}, FF {(betas['exret'] - betas['exp_ff']).std():.4f}")

**Comment:** We get betas for 154,802 bank-days and all 59 banks (a bank needs 300 trading days of history first). The median CAPM beta is 1.26 for US banks and 1.31 for European banks. In the Fama-French five-factor model both load strongly on HML (0.99 and 0.86), so banks behave like value stocks, and the market beta of European banks drops to 0.89 once the other four factors are included. Both also load negatively on RMW (−0.91 for US banks, −1.54 for European banks), so they behave like weakly profitable firms, and European banks load negatively on CMA (−0.50, against −0.05 for US banks), like aggressive investors. The extra factors reduce the daily abnormal-return volatility from 2.32% (CAPM) to 2.18% (FF5).

**Effect of the USD conversion:** converting returns to USD puts returns and factors in the same currency. It affects the European banks (38 of the 59); US banks are unchanged. 

# Part C: Event study 

## C1. Event study: CAPM and cumulative abnormal returns
We follow Krueger (2015). For each kept campaign event, we estimate a market model for the targeted bank just before the event, use it to predict the bank's "normal" return around the event, and measure how far the actual return deviates from it.

In [ ]:
ev = events[events["keep"]].copy()

ev = ev.merge(
    panel[["ds_name", "date", "t"]], 
    left_on=["ds_name", "trade_date"],
    right_on=["ds_name", "date"],
    how="left",  
    suffixes=("", "_panel"),
)
ev = ev.rename(columns={"t": "t0"})

In [ ]:
print(f"{len(ev):,} events kept")
print(f"{ev['t0'].isna().sum()} without a matching trading day in the panel")
print(f"{(ev['t0'] < 300).sum()} with less than 300 days of history before t0")

In [ ]:
miss = ev[ev["t0"].isna()]
span = panel.groupby("ds_name")["date"].agg(first="min", last="max")
miss = miss.join(span, on="ds_name")

outside = (miss["trade_date"] < miss["first"]) | (miss["trade_date"] > miss["last"])
print(f"{outside.sum()} outside the bank's price history (not yet listed / delisted)")
print(f"{(~outside).sum()} inside the history but no return that day (holiday / removed return)")
print(f"{miss['ds_name'].nunique()} distinct banks concerned")

**Locating the event day:** Trading days (`t`) are numbered bank by bank in Part B, so all windows are measured in trading days. We find the event day t0 by matching each kept event to the panel on bank and trade date. Of the 1,363 kept events, 50 (7 banks) fall outside the bank's price history: the bank was not yet listed, or no longer listed, on the event date, so no market reaction can be measured. No event is lost to holidays or removed returns. A further 12 events have fewer than 300 trading days before t0, so the estimation window [t0−300, t0−50] cannot be filled. Dropping both leaves 1,301 events (95% of the kept sample) on 55 banks.

In [ ]:
ev = ev[ev["t0"].notna() & (ev["t0"] >= 300)].copy()
ev["t0"] = ev["t0"].astype(int)          #t0 was float because of the NaNs
print(f"{len(ev):,} events for the event study")

**Method:** All windows are measured in the bank's trading days (`t`), not calendar days, and all returns are in USD.

- **Estimation window** [t0−300, t0−50] (251 days). We use the parameters estimated in Part B, read at t0 and held fixed over the whole event window, so the benchmark only uses information from before the event:
  $r_{i,t} - r_{f,t} = \alpha_i + \beta_i \,(r_{m,t} - r_{f,t}) + \varepsilon_{i,t}$.
  The 50-day gap keeps any pre-event information leakage out of the estimation.
- **Abnormal return** on each day of the event window:
  $AR_{i,t} = (r_{i,t} - r_{f,t}) - (\hat\alpha_i + \hat\beta_i \,(r_{m,t} - r_{f,t}))$.
- **CAR** = sum of the daily ARs over the window. We also keep the event-day abnormal return `ar0`.

`run_event_study` takes the window (`pre`, `post`) and the model as arguments. Our main specification is the CAPM on [t0−10, t0+10] (21 days); we also run [t0−1, t0+1] and [t0−5, t0+5], and a Fama-French model. For each event, it also computes the residual volatility `sigma` and the share of positive residuals `pos_share` on the estimation window, which the significance tests need, and `n_other`, the number of other campaigns on the same bank inside the event window.

Because `t0 ≥ 300` and `t` has no gaps, every estimation window has exactly 251 days. The function also checks that the event window is complete, which matters only if a bank's price series ends before t0+10; this never happens here, so all 1,301 events get a CAR.

In [ ]:
# all campaign days of each bank, in trading days, including those dropped by the overlap rule
camp = events.merge(panel[["ds_name", "date", "t"]], left_on=["ds_name", "trade_date"],
                    right_on=["ds_name", "date"], how="inner", suffixes=("", "_panel"))
campaign_t = camp.groupby("ds_name")["t"].apply(lambda s: np.sort(s.unique()))

In [ ]:
MODELS = {"capm": ["mkt_rf"], "ff": FACTORS}


def run_event_study(ev, panel, betas, pre=10, post=10, model="capm"):

    factors = MODELS[model]
    cols = [f"{model}_alpha"] + [f"{model}_{f}" for f in factors]
    data = ev.merge(betas[["ds_name", "t", *cols]], left_on=["ds_name", "t0"], right_on=["ds_name", "t"], how="left")

    rows = []
    for _, e in data.iterrows():
        bank = panel[panel["ds_name"] == e["ds_name"]]
        win = bank[(bank["t"] >= e["t0"] - pre) & (bank["t"] <= e["t0"] + post)].copy()

        if len(win) < pre + post + 1:
            continue

        def predict(d):
            return e[f"{model}_alpha"] + sum(e[f"{model}_{f}"] * d[f] for f in factors)

        win["ar"] = win["exret"] - predict(win)

        est = bank[(bank["t"] >= e["t0"] - 300) & (bank["t"] <= e["t0"] - 50)]
        resid = est["exret"] - predict(est)

        rows.append({
            "ds_name": e["ds_name"],
            "trade_date": e["trade_date"],
            "alpha": e[f"{model}_alpha"],
            "beta": e[f"{model}_{factors[0]}"],
            "car": win["ar"].sum(),
            "ar0": win["ar"].iloc[pre], 
            "n_big": int(win["big"].sum()),
            "ar_path": win["ar"].to_numpy(),
            "sigma": resid.std(ddof=len(factors) + 1),     # residual volatility, for Boehmer
            "pos_share": (resid > 0).mean(),               # share of positive ARs, for Cowan
            "r2": 1 - resid.var() / est["exret"].var(),
            "sign": e["sign"],
            "uids": e["uids"],
            "n_other": int(((campaign_t[e["ds_name"]] >= e["t0"] - pre) & (campaign_t[e["ds_name"]] <= e["t0"] + post)
                            & (campaign_t[e["ds_name"]] != e["t0"])).sum()),   # other campaigns inside the window
        })

    return pd.DataFrame(rows)

In [ ]:
studies = {}
for model in ["capm", "ff"]:
    for k in [1, 5, 10]:
        studies[(model, k)] = run_event_study(ev, panel, betas, pre=k, post=k, model=model)

summary = pd.DataFrame({key: s["car"].describe() for key, s in studies.items()}).T
print(summary[["count", "mean", "50%", "std", "min", "max"]].round(4))
print(f"Correlation CAPM / FF on [-10,+10]: {studies[('capm', 10)]['car'].corr(studies[('ff', 10)]['car']):.3f}")
print(studies[("capm", 10)][["sigma", "pos_share"]].describe().round(4))

In [ ]:
# diagnostics of the market model on the estimation window, CAPM only

from statsmodels.stats.diagnostic import het_breuschpagan

data = ev.merge(betas[["ds_name", "t", "capm_alpha", "capm_mkt_rf"]], left_on=["ds_name", "t0"], right_on=["ds_name", "t"], how="left")

rows = []
for _, e in data.iterrows():
    bank = panel[panel["ds_name"] == e["ds_name"]]
    est = bank[(bank["t"] >= e["t0"] - 300) & (bank["t"] <= e["t0"] - 50)]

    x, y = est["mkt_rf"], est["exret"]
    resid = y - (e["capm_alpha"] + e["capm_mkt_rf"] * x)

    half = len(est) // 2
    x1, y1 = x.iloc[:half], y.iloc[:half]
    x2, y2 = x.iloc[half:], y.iloc[half:]

    rows.append({
        "ds_name": e["ds_name"],
        "trade_date": e["trade_date"],
        "dw": (resid.diff() ** 2).sum() / (resid ** 2).sum(),
        "bp_p": het_breuschpagan(resid, sm.add_constant(x))[1],
        "beta_1": x1.cov(y1) / x1.var(),
        "beta_2": x2.cov(y2) / x2.var(),
    })

cars = studies[("capm", 10)].merge(pd.DataFrame(rows), on=["ds_name", "trade_date"])

In [ ]:
print(f"{len(cars):,} events with a CAR")
print(cars[["beta", "car", "ar0"]].describe().round(4))

In [ ]:
low_dw = cars["dw"] < 1.5
cars["beta_diff"] = cars["beta_2"] - cars["beta_1"]
print(f"{low_dw.mean():.1%} of events have a Durbin-Watson < 1.5 (autocorrelated residuals)")
bp_events = cars["bp_p"] < 0.05
print(f"{bp_events.mean():.1%} of events have a Breusch-Pagan p-value < 0.05 (heteroskedastic residuals)")
print(cars[["r2", "dw", "beta_diff"]].describe().round(4))
print(f"Correlation between first- and second-half betas: {cars['beta_1'].corr(cars['beta_2']):.2f}")
print(cars.groupby("ds_name")["r2"].median().nsmallest(6).round(3))

**Diagnostics of the market model:** The market factor explains a median 46% of daily bank returns (R²). The lowest R² belong to Umweltbank, Van Lanschot Kempen, Permanent TSB and AIB, three of which also have many zero-return days. Residuals show little first-order autocorrelation (mean Durbin-Watson 2.10; only 0.4% of events below 1.5). The Breusch-Pagan test rejects homoskedasticity for 18% of events (against 5% expected by chance): this does not bias α and β, but it means that our significance tests should not rely on a constant variance. Betas estimated on the two halves of the estimation window show no systematic drift (mean difference −0.03) but vary substantially across the two halves (standard deviation 0.39, correlation 0.56), consistent with changing risk during the crisis years. We therefore check below that our results hold with a market-adjusted model (β = 1), which does not require estimating β.

In [ ]:
low = cars[cars["beta"] < 0.3]
print(f"Betas: mean {cars['beta'].mean():.2f}, median {cars['beta'].median():.2f}")
print(f"{len(low)} events with beta < 0.3:")
for bank, n in low["ds_name"].value_counts().items():
    print(f"{bank}: {n} events, {zero[bank]:.1%} zero-return days")

**Result:** The mean estimated beta is 1.36 and the median 1.34, so on average banks move more than the market. Only 3 events, all on Umweltbank, have a beta below 0.3. Umweltbank is a small German bank with a zero return on 12.6% of its trading days, and infrequent price changes bias betas towards zero.

In [ ]:
print(f"{(cars['car'] < -0.5).sum()} events with CAR < -50%, {(cars['car'] > 0.3).sum()} with CAR > +30%")
print(cars.nsmallest(5, "car")[["ds_name", "trade_date", "car", "beta"]].round({"car": 3, "beta": 3}))
print(cars.nlargest(5, "car")[["ds_name", "trade_date", "car", "beta"]].round({"car": 3, "beta": 3}))

**Extreme CARs:** A handful of events have very large CARs. In the cases below, a major bank-specific or sector-wide shock falls inside the event window, so these CARs are most likely driven by that shock rather than by the NGO campaign:

- **Bankia, May 2013 (CAR ≈ −139%):** the state recapitalisation by the FROB almost wiped out existing shareholders and forcibly converted retail hybrid debt into shares; when trading resumed, these new shareholders sold massively. A CAR below −100% does not mean the share lost more than its value: the CAR sums daily abnormal returns instead of compounding them, which overstates losses when daily moves are very large.
- **Monte dei Paschi, December 2016 (−32%):** the private recapitalisation failed, trading was suspended and the Italian state stepped in.
- **Barclays, July 2012 (−25%):** the LIBOR scandal, with a settlement with regulators and the resignation of its CEO.
- **Bankinter, May 2012 (−30%) and UniCredit, January 2016 (−24%):** periods of stress for Spanish and Italian banks.
- **Positive extremes, +30% to +41%:** mostly rebounds after periods of stress, e.g. KBC in January 2012, when European bank stocks rallied after the ECB's long-term refinancing operations, and Deutsche Bank in October 2016, recovering from fears of a $14bn fine from the US Department of Justice.

We draw two lessons. First, a few events dominate the mean, so we winsorise CARs at 1%/99% and also report medians. Second, four of the five most negative CARs are Spanish or Italian banks, and the market factor does not capture national or bank-sector shocks; a bank-sector benchmark would be a natural extension.

In [ ]:
def winsorise(s, col="car"):
    lo, hi = s[col].quantile([0.01, 0.99])
    return s.assign(car_w=s[col].clip(lo, hi))

In [ ]:
cars = winsorise(cars)
print(f"CARs winsorised at 1%/99%: [{cars['car_w'].min():.1%}, {cars['car_w'].max():.1%}]")
print(f"{(cars['n_big'] > 0).sum()} events with a >50% daily move in the event window")

In [ ]:
cars = cars.merge(
    ev[["ds_name", "trade_date", "sentiment", "negative", "prominence",
        "ngo_power", "n_targets", "climate", "mass", "region"]],
    on=["ds_name", "trade_date"], how="left",
)
print(cars.shape)
cars["ds_name"].nunique()

In [ ]:
# other windows and models, then the handoff file (one row per event, without the daily path)
extra = {"car_1": ("capm", 1), "car_5": ("capm", 5),
         "car_ff": ("ff", 10), "car_ff_1": ("ff", 1), "car_ff_5": ("ff", 5)}

for name, key in extra.items():
    cars = cars.merge(studies[key][["ds_name", "trade_date", "car"]].rename(columns={"car": name}),
                      on=["ds_name", "trade_date"], how="left")

cars.drop(columns="ar_path").to_csv(OUT / "cars.csv", index=False)
print(cars.shape, cars["ds_name"].nunique())

**Output of Part C:** `cars` holds one row per event (1,301 events, 55 banks). CARs are winsorised at 1%/99% (bounds [−17.7%, +21.1%]). Next, we test whether they differ from zero, separately for negative and positive campaigns.

In [ ]:
def clustered_t(y, groups):
    # regressing y on a constant gives its mean; the cluster SE allow correlation within a group
    fit = sm.OLS(y.to_numpy(), np.ones(len(y))).fit(cov_type="cluster", cov_kwds={"groups": pd.factorize(groups)[0]})
    return fit.tvalues[0]

def car_tests(s, pre, post, col="car_w"):
    y, n = s[col], len(s)

    scar = y / (s["sigma"] * np.sqrt(pre + post + 1))       # CAR in units of its expected volatility
    p = s["pos_share"].mean()                               # share of positive ARs in normal times

    return pd.Series({
        "n": n,
        "mean": y.mean(),
        "median": y.median(),
        "t_plain": y.mean() / (y.std() / np.sqrt(n)),
        "t_boehmer": scar.mean() * np.sqrt(n) / scar.std(),
        "z_cowan": ((y > 0).sum() - n * p) / np.sqrt(n * p * (1 - p)),
        "t_date": clustered_t(y, s["trade_date"]),
        "t_month": clustered_t(y, s["trade_date"].dt.to_period("M")),
    })

In [ ]:
def test_table(studies, keep=None, col="car_w"):
    rows = {}
    for (model, k), st in studies.items():
        w = winsorise(st)
        if keep is not None:
            w = w[keep(w)]
        for sign in ["neg", "pos"]:
            rows[(model, k, sign)] = car_tests(w[w["sign"] == sign], k, k, col)
    out = pd.DataFrame(rows).T
    out.index.names = ["model", "window", "sign"]
    return out


results = test_table(studies)
clean = test_table(studies, keep=lambda w: w["n_other"] == 0)

print(pd.Series({k: (s["n_other"] > 0).mean() for k, s in studies.items()}).round(3))
print(results.round(3).to_string())
print(clean.round(3).to_string())
m, t = results.loc[("capm", 10, "neg"), ["mean", "t_month"]]
se = m / t
print(f"95% CI of the mean CAR (month-clustered): [{m - 1.96 * se:.1%}, {m + 1.96 * se:.1%}]")

**Significance tests:** We test whether the mean CAR differs from zero, separately for negative and positive campaigns, on winsorised CARs. A plain t-test assumes independent events, so we report several tests:

- **Boehmer et al.:** scales each CAR by the bank's own volatility and uses the observed dispersion around the event, so it allows for higher volatility on event days.
- **Cowan's sign test:** compares the share of positive CARs with its normal share in the estimation window, so extreme CARs do not matter.
- **t-tests clustered by date and by month:** allow CARs to be correlated across events. Windows last up to 21 days, so events close in time share the same shocks, and the month-clustered test is our most cautious one.

**Results:** For negative campaigns, our main specification (CAPM, [−10,+10], 1,054 events) gives a mean CAR of −0.4% and a median of −0.7%. Boehmer (−2.44) is significant at 5%, and Cowan (−1.91) is close, but the effect is not significant once we allow for correlation across events (−1.38 by date, −0.99 by month). The 95% confidence interval from the month-clustered test is about [−1.1%, +0.4%], so a small negative effect cannot be ruled out, but we find no robust evidence of one. With the Fama-French five-factor model, the mean is −0.3% (Boehmer −1.70, −1.28 by month), also not significant. This differs from Krueger (2015), who finds a strong negative reaction to negative events.

For positive campaigns (162 events), the mean CAR on [−1,+1] is −0.6% (−3.38 Boehmer, −2.72 by date, −2.84 by month), but the sign test is weaker (−1.56). On longer windows, the effect is not significant with clustering. A negative reaction to good news is in line with Krueger's weakly negative result, but we do not test why it happens here.

These pooled results hide a difference between regions: see the bank-benchmark extension at the end of Part C, where negative campaigns have a short-lived effect for European banks only.

In [ ]:
tests = {"t_plain": "Plain t", "t_boehmer": "Boehmer", "z_cowan": "Cowan sign",
         "t_date": "Clustered\nby date", "t_month": "Clustered\nby month"}
cases = {"Negative campaigns, [−10,+10]": ("capm", 10, "neg"), "Positive campaigns, [−1,+1]": ("capm", 1, "pos")}

x, w = np.arange(len(tests)), 0.38
fig, ax = plt.subplots(figsize=(8, 3.6))
for j, (label, key) in enumerate(cases.items()):
    ax.bar(x + (j - 0.5) * w, results.loc[key, list(tests)], width=w, color=C[key[2]],
           label=f"{label} (n = {int(results.loc[key, 'n'])})")

for level in (-1.96, 1.96):
    ax.axhline(level, color=C["grey"], ls="--", lw=1)
ax.axhline(0, color="black", lw=0.8)
ax.text(-0.45, 1.96, "5% threshold (±1.96)", ha="left", va="bottom", fontsize=9, color=C["grey"])

ax.set_xticks(x, list(tests.values()), fontsize=9)
ax.set_ylabel("Test statistic")
style([ax])
fig.legend(*ax.get_legend_handles_labels(), loc="lower center", ncol=2, frameon=False,
           bbox_to_anchor=(0.5, -0.14), fontsize=9)
fig.suptitle("Figure 5: The negative effect depends on the test, the positive one holds under most tests",
             x=0.01, y=1.02, ha="left", fontsize=12, fontweight="bold")
fig.text(0.01, -0.22, "All banks, CAPM CARs winsorised at 1%/99%. A bar beyond a dashed line is significant at 5%.",
         fontsize=8, color=C["grey"])
plt.show()

**Figure 5:** For negative campaigns on [−10,+10], only Boehmer's test is significant at 5% (−2.44). The plain t-test (−1.84) and Cowan's sign test (−1.91) are just short of the threshold, and the statistic falls to −1.38 once errors are clustered by date and to −0.99 by month. The apparent effect therefore relies on treating events as independent, which they are not: many campaigns fall on the same days or in the same weeks. For positive campaigns on [−1,+1], four of the five tests are significant, including both clustered ones (−2.72 by date, −2.84 by month). Only the sign test is not (−1.72), which suggests that part of the effect comes from a minority of large negative CARs rather than a general shift.

In [ ]:
# market-adjusted model: alpha = 0 and beta = 1
MODELS["mkt"] = ["mkt_rf"]
betas["mkt_alpha"], betas["mkt_mkt_rf"] = 0.0, 1.0
for k in [1, 5, 10]:
    studies[("mkt", k)] = run_event_study(ev, panel, betas, pre=k, post=k, model="mkt")

stale = zero[zero > 0.10].index
print(f"{len(stale)} banks with more than 10% zero-return days: {list(stale)}")

checks = {
    "baseline": test_table(studies),
    "clean": test_table(studies, keep=lambda w: w["n_other"] == 0),
    "raw": test_table(studies, col="car"),
    "no_stale": test_table(studies, keep=lambda w: ~w["ds_name"].isin(stale)),
}

In [ ]:
pos = winsorise(studies[("capm", 1)])
pos = pos[pos["sign"] == "pos"].sort_values("car_w")

print(f"{(pos['car_w'] < 0).mean():.1%} of positive campaigns have a negative CAR on [-1,+1]")
print(pos[["ds_name", "trade_date", "car_w"]].head(5).round({"car_w": 3}).to_string())
print(pos["trade_date"].value_counts().head(5).to_string())
print(pos.groupby("ds_name")["car_w"].agg(["size", "mean", "sum"]).sort_values("sum").head(5).round(3).to_string())

print(car_tests(pos.iloc[5:], 1, 1)[["n", "mean", "t_month"]].round(4).to_string())

**Positive campaigns on [−1,+1]:** The effect is not driven by a few events: without the five most negative CARs, the mean is −0.36% (t_month −1.93), and 57% of CARs are negative. We read it with caution: it was not our starting hypothesis, it rests on 162 events, and 24 of them fall on just four dates.

In [ ]:
keys = [("capm", 10, "neg"), ("ff", 10, "neg"), ("capm", 1, "pos"), ("ff", 1, "pos"), ("mkt", 1, "pos")]
print(pd.concat({name: t.loc[keys, ["mean", "t_month"]] for name, t in checks.items()}, axis=1).round(4).to_string())

**Robustness:** The table shows our key cases under each check: dropping events with another campaign in the window (`clean`, 45% of events on [−10,+10]), raw instead of winsorised CARs (`raw`), and dropping banks with many zero-return days (`no_stale`). For negative campaigns, the mean CAR stays negative but never significant. The Fama-French five-factor model gives a similar mean (−0.33% against −0.39% under CAPM) and is also not significant (month-clustered t = −1.28 against −0.99). For positive campaigns on [−1,+1], the mean stays around −0.5% and its month-clustered t-statistic exceeds 2 in absolute value under every check, including the β = 1 model, although the sign test is weaker.

The bank-sector benchmark at the end of Part C is a further check: it removes shocks common to all banks, and it confirms that the positive-campaign effect is limited to [−1,+1].

In [ ]:
# data check: bank-years where more than half of the days have a zero return
p = panel[panel["date"] >= "2010-01-01"].assign(zero=lambda d: d["ret_local"].abs() < 1e-12)
by_year = p.groupby(["ds_name", p["date"].dt.year])["zero"].mean()
print(by_year[by_year > 0.5].round(2).to_string())

**Data check:** Monte dei Paschi's return series is almost entirely frozen from 2017 to 2020. This does not affect our results, because its only event in the sample is on 9 December 2016.

In [ ]:
def build_ev(gap):
    e = events[overlap_rule(events, gap)].merge(panel[["ds_name", "date", "t"]], left_on=["ds_name", "trade_date"],
                                                right_on=["ds_name", "date"], how="inner", suffixes=("", "_panel"))
    e = e.rename(columns={"t": "t0"})
    return e[e["t0"] >= 300].copy()


rows = {}
for gap in [30, 45, 60, 90]:
    e = build_ev(gap)
    for model in ["capm", "ff"]:
        w = winsorise(run_event_study(e, panel, betas, model=model))
        for sign in ["neg", "pos"]:
            rows[(gap, model, sign)] = car_tests(w[w["sign"] == sign], 10, 10)

gaps = pd.DataFrame(rows).T
gaps.index.names = ["gap", "model", "sign"]
print(gaps[["n", "mean", "median", "t_boehmer", "t_month"]].round(4).to_string())

**Other overlap gaps:** On [−10,+10], gaps of 30 to 90 days give a mean CAPM CAR between −0.33% and −0.51% for negative campaigns, with t_month between −0.76 and −1.11. A longer gap does not remove the other campaigns from the window: it only changes which events we keep.

## Extension: a bank-sector benchmark

In [ ]:
# leave-one-out bank index: each day, the equal-weight return of the other banks in the same region
g = panel.groupby(["region", "date"])["ret"]
n_banks = g.transform("count")
panel["bank_mkt"] = (g.transform("sum") - panel["ret"]) / (n_banks - 1)
panel["bank_rf"] = panel["bank_mkt"] - panel["rf"]

print(n_banks.groupby(panel["region"]).describe()[["min", "50%", "max"]])
print(panel.groupby("region")[["bank_rf", "mkt_rf"]].corr().round(2))
print(panel.groupby("region")[["bank_rf", "mkt_rf"]].std().round(4))

**Bank index:** Each day, a bank's benchmark is the equal-weight return of the other banks in the same region (US or Europe), so a bank is never compared with itself. The index is correlated at 0.80–0.82 with the market factor, so the market explains about two thirds of its daily variance, and it is much more volatile (2.2–2.4% a day against 1.3%). The rest includes shocks common to banks, which this benchmark is meant to capture, and some noise.

In [ ]:
# when does a region have few banks?
print(panel[n_banks < 10].groupby("region")["date"].agg(["min", "max", "count"]))

# rolling alpha and beta on the bank index, same window [t-300, t-50] as Part B
parts = []
for _, d in panel.groupby("ds_name"):
    parts.append(pd.concat([d[["ds_name", "t"]], rolling_params(d, ["bank_rf"], "bank")], axis=1))
bank_params = pd.concat(parts, ignore_index=True)

betas = betas.drop(columns=["bank_alpha", "bank_bank_rf"], errors="ignore").merge(
    bank_params[["ds_name", "t", "bank_alpha", "bank_bank_rf"]], on=["ds_name", "t"], how="left")
MODELS["bank"] = ["bank_rf"]

print(betas[["capm_mkt_rf", "bank_bank_rf"]].describe().round(2))
print(panel.groupby("ds_name")["date"].max().value_counts().head())

**Bank betas:** Most bank series end on 28 April 2020 (56 of 59 banks), so Europe has fewer than 10 banks only from 29 April to 30 September 2020, after the end of our event windows. Betas on the bank index average 0.96, against 1.31 on the market factor, and are less dispersed (standard deviation 0.29 against 0.43): a bank moves roughly like the other banks.

In [ ]:
for k in [1, 5, 10]:
    studies[("bank", k)] = run_event_study(ev, panel, betas, pre=k, post=k, model="bank")

bank = test_table({key: s for key, s in studies.items() if key[0] == "bank"})
print(bank.round(3).to_string())

# do the extremes shrink once bank-sector shocks are removed?
cmp = studies[("capm", 10)][["ds_name", "trade_date", "car"]].merge(
    studies[("bank", 10)][["ds_name", "trade_date", "car"]], on=["ds_name", "trade_date"], suffixes=("_capm", "_bank"))
print(f"Correlation CAPM / bank CARs on [-10,+10]: {cmp['car_capm'].corr(cmp['car_bank']):.2f}")
print(cmp.nsmallest(5, "car_capm").round({"car_capm": 3, "car_bank": 3}).to_string())
print(cmp.nlargest(5, "car_capm").round({"car_capm": 3, "car_bank": 3}).to_string())

**Results with the bank benchmark:** CARs from the two models are correlated at 0.85 on [−10,+10]. Removing shocks common to all banks shrinks the sector-wide extremes, such as Barclays and Deutsche Bank in October 2016 (+31% and +30% with the CAPM, +9% and +13% here) or KBC in January 2012, while bank-specific crises such as Bankia, Monte dei Paschi and Bankinter stay extreme. For negative campaigns, the pooled result does not change: there is no robust effect. For positive campaigns, the effect on [−1,+1] survives (−0.4%, t_month −2.48), but on [−5,+5] and [−10,+10] it disappears, so the longer-window CAPM results reflected sector-wide moves. One caveat: when a campaign targets several banks on the same day, the other targeted banks are part of the index, so this benchmark may understate the effect of multi-bank campaigns.

In [ ]:
short = {"royal bank of sctl.gp.": "RBS", "banca monte dei paschi": "MPS", "bankinter 'r'": "Bankinter",
         "deutsche bank": "Deutsche Bank", "kbc group": "KBC"}
region_of = panel.groupby("ds_name")["region"].first()
pts = cmp.assign(region=cmp["ds_name"].map(region_of), x=100 * cmp["car_capm"], y=100 * cmp["car_bank"])

lim = 45
fig, ax = plt.subplots(figsize=(6.5, 5.2))
for r, (name, col) in REG.items():
    d = pts[pts["region"] == r]
    ax.scatter(d["x"], d["y"], s=10, alpha=0.45, color=col, label=name)
ax.plot([-lim, lim], [-lim, lim], color=C["grey"], ls="--", lw=1)
ax.axhline(0, color="black", lw=0.6)
ax.axvline(0, color="black", lw=0.6)

extremes = pd.concat([pts.nsmallest(5, "x"), pts.nlargest(5, "x")])

for _, e in extremes[extremes["x"].abs() < lim].iterrows():
    left = -25 < e["x"] < 0       
    ax.annotate(f"{short.get(e['ds_name'], e['ds_name'].title())} {e['trade_date']:%m/%Y}", (e["x"], e["y"]),
                xytext=(-5 if left else 5, 3), textcoords="offset points", fontsize=7.5, ha="right" if left else "left")

ax.set_xlim(-lim, lim)
ax.set_ylim(-lim, lim)
ax.set_xlabel("CAR with the CAPM, [−10,+10] (%)")
ax.set_ylabel("CAR with the bank index, [−10,+10] (%)")
ax.legend(frameon=False, loc="upper left", fontsize=9)
style([ax])
fig.suptitle("Figure 6: The bank index removes sector-wide shocks", x=0.01, y=1.0, ha="left", fontsize=12, fontweight="bold")
fig.text(0.01, -0.04, "One point per event. The dashed line is where both models agree. "
         "Bankia (May 2013, about −139% with both models) is outside the plot.", fontsize=8, color=C["grey"])
plt.show()

**Figure 6:** Each point compares an event's CAR under the CAPM (horizontal axis) and under the bank index (vertical axis). Most events lie close to the diagonal, and the two measures are correlated at 0.85. The large CARs, however, tend to shrink towards zero: the cloud is flatter than the diagonal on both sides. These are sector-wide moves that the CAPM attributes to the bank. Deutsche Bank and Barclays in October 2016 fall from about +30% to +13% and +9%, KBC in January 2012 from +41% to +26%, and UniCredit in January 2016 and Barclays in July 2012 roughly halve. Bank-specific crises stay on or below the diagonal: Bankinter (−30% and −31%), Monte dei Paschi (−32% and −38%) and Bankia in May 2013 (about −139% with both, outside the plot). RBS in April 2010 even rises (+33% to +39%), as its stock outperformed the other banks.

In [ ]:
region_of = panel.groupby("ds_name")["region"].first()
main = {key: s for key, s in studies.items() if key[0] in ("capm", "bank")}

for r, name in [("US", "US banks"), ("EU", "European banks")]:
    t = test_table(main, keep=lambda w, r=r: w["ds_name"].map(region_of) == r)
    print(f"\n{name}")
    print(t[["n", "mean", "median", "t_boehmer", "z_cowan", "t_date", "t_month"]].round(3).to_string())

**Results by region:** We split the tests by region, for the CAPM and the bank benchmark (CARs are winsorised on the full sample before the split). This split was not planned before looking at the results, so we treat it as exploratory. For US banks, there is no effect for any window or model. For European banks, negative campaigns lower returns by about 0.2–0.3% on [−1,+1], significant with month-clustered errors under both the CAPM (t_month −2.39) and the bank benchmark (−2.74). On longer windows, the effect is not significant once clustered, and the negative reaction to positive campaigns is also entirely European.

In [ ]:
eu = lambda w: w["ds_name"].map(region_of) == "EU"
eu_checks = {
    "baseline": test_table(studies, keep=eu),
    "clean": test_table(studies, keep=lambda w: eu(w) & (w["n_other"] == 0)),
    "raw": test_table(studies, keep=eu, col="car"),
    "no_stale": test_table(studies, keep=lambda w: eu(w) & ~w["ds_name"].isin(stale)),
}
keys_eu = [("capm", 1, "neg"), ("ff", 1, "neg"), ("mkt", 1, "neg"), ("bank", 1, "neg")]
print(pd.concat({name: t.loc[keys_eu, ["n", "mean", "t_month"]] for name, t in eu_checks.items()}, axis=1).round(4).to_string())

**Robustness of the European result:** The effect on [−1,+1] holds when we drop events with other campaigns in the window, use raw CARs or drop banks with many zero-return days, under both the CAPM and the bank benchmark. It is weaker with β = 1 (t_month between −1.69 and −2.16) and not significant with Fama-French factors, where the mean is about halved. The pooled results therefore mix a short-lived effect for European banks with no effect for US banks.

In [ ]:
def mean_ci(y, groups):
    # mean and 95% confidence interval, with errors clustered by month
    fit = sm.OLS(y.to_numpy(), np.ones(len(y))).fit(cov_type="cluster", cov_kwds={"groups": pd.factorize(groups)[0]})
    m, se = fit.params[0], fit.bse[0]
    return 100 * m, 100 * (m - 1.96 * se), 100 * (m + 1.96 * se)


labels = {"capm": "CAPM", "ff": "Fama-French", "mkt": "β = 1", "bank": "Bank index"}
rows = []
for model, label in labels.items():
    w = winsorise(studies[(model, 1)])
    w["region"] = w["ds_name"].map(region_of)
    for (r, sign), d in w[w["sign"] != "neu"].groupby(["region", "sign"]):
        m, lo, hi = mean_ci(d["car_w"], d["trade_date"].dt.to_period("M"))
        rows.append({"model": label, "region": r, "sign": sign, "mean": m, "lo": lo, "hi": hi, "n": len(d)})
ci = pd.DataFrame(rows)

fig, axes = plt.subplots(1, 2, figsize=(11, 4.6), sharex=True, sharey=True)
y = np.arange(len(labels))
for ax, (r, title) in zip(axes, [("US", "A. US banks"), ("EU", "B. European banks")]):
    n = ci[ci["region"] == r].groupby("sign")["n"].first()
    for sign, shift, name in [("neg", 0.18, "Negative campaigns"), ("pos", -0.18, "Positive campaigns")]:
        d = ci[(ci["region"] == r) & (ci["sign"] == sign)].set_index("model").loc[list(labels.values())]
        ax.errorbar(d["mean"], y + shift, xerr=[d["mean"] - d["lo"], d["hi"] - d["mean"]],
                    fmt="o", ms=6, color=C[sign], capsize=3, lw=1.6, label=name)
    ax.axvline(0, color="black", lw=0.8)
    ax.set_yticks(y, list(labels.values()), fontsize=10)
    ax.set_title(f"{title} ({n['neg']} negative, {n['pos']} positive)", loc="left", fontweight="bold", fontsize=11)
    ax.set_xlabel("Mean CAR on [−1,+1] (%)")
axes[0].invert_yaxis()
style(axes)

handles, names = axes[0].get_legend_handles_labels()
fig.legend(handles, names, loc="lower center", ncol=2, frameon=False, bbox_to_anchor=(0.5, -0.07), fontsize=10)
fig.suptitle("Figure 7: European bank stocks fall slightly around NGO campaigns, US ones do not",
             x=0.01, y=1.03, ha="left", fontsize=13.5, fontweight="bold")
fig.text(0.01, -0.13, "Mean CAR over [−1,+1], winsorised at 1%/99%, with 95% confidence intervals clustered by month. "
         "An interval that does not cross zero is significant at 5%.", fontsize=8.5, color=C["grey"])
plt.show()
print(ci.round(2).to_string())

**Figure 7:** Each point is the mean CAR on [−1,+1] with its 95% confidence interval, clustered by month, for the four benchmarks. *US banks:* every interval crosses zero, for negative and positive campaigns alike. The point estimates for negative campaigns are even slightly positive (+0.12% to +0.22%). *European banks:* negative campaigns lower returns by 0.16% to 0.29%. The interval excludes zero with the CAPM (−0.29%, [−0.53%, −0.05%]) and the bank index (−0.22%, [−0.38%, −0.06%]), touches it with β = 1 and crosses it with Fama-French (−0.16%, [−0.36%, +0.04%]). Positive campaigns lower European returns by more, 0.46% to 0.69%, and the interval excludes zero under all four benchmarks, but it is wide because there are only 125 events. The figure summarises our main finding: there is no reaction for US banks, and a small, short-lived drop for European banks, after both negative and, surprisingly, positive campaigns.

# Task 6: Do investors react to NGO campaigns, and which campaigns does it depend on?

Follows Krüger (2015). Works from the handoff files of `Hackathon_aligned.ipynb`, so nothing here touches the main notebook. The returns are already in USD, so this notebook doesn't need `fx_usd.csv`.

**Hypotheses, fixed before testing:**
1. Negative campaigns lower returns.
2. Positive campaigns raise them.
3. Bank-specific campaigns matter more than multi-firm ones.
4. Prominent campaigns and high-reach NGOs matter more.
5. Climate campaigns react differently after COP21.

## 1. Setup

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
from pathlib import Path

HANDOFF = OUT                      # copied from the output of Hackathon_aligned.ipynb
EST_START, EST_END = 300, 50                                 # estimation window [t-300, t-50], same as B and C
PRE, POST = 10, 10
COP21 = pd.Timestamp("2015-12-12")                           # Paris Agreement


panel = pd.read_csv(HANDOFF / "panel_returns.csv", parse_dates=["date"])
betas = pd.read_csv(HANDOFF / "betas_daily.csv", parse_dates=["date"])
cars = pd.read_csv(HANDOFF / "cars.csv", parse_dates=["trade_date"])    # C's results, used as a check below
cars["post"] = cars["trade_date"] > COP21
cars["month"] = cars["trade_date"].dt.to_period("M").astype(str)
print(len(cars), cars["sign"].value_counts().to_dict())
print(f"{len(panel):,} bank-days | {panel['ds_name'].nunique()} banks | {len(cars):,} CARs")

## 4. Main table (H1, H2)

In [ ]:
for col in ["car", "car_1", "car_5"]:
    lo, hi = cars[col].quantile([0.01, 0.99])
    cars[col + "_win"] = cars[col].clip(lo, hi)

d = cars[cars["sign"] == "neg"]
y = d["car_win"]
n = len(d)
print(n, 100 * y.mean(), 100 * y.median())


In [ ]:
def group_tests(d, col="car_win", width=21):
    y, n = d[col], len(d)

    scar = y / (d["sigma"] * np.sqrt(width))
    t_boehmer = scar.mean() * np.sqrt(n) / scar.std()

    p = d["pos_share"].mean()
    z_cowan = ((y > 0).sum() - n * p) / np.sqrt(n * p * (1 - p))

    t_date = clustered_t(y, d["trade_date"])
    t_month = clustered_t(y, d["month"])
    
    return {"N": n, "Mean (%)": 100 * y.mean(), "Median (%)": 100 * y.median(), 
            "t_Boehmer": t_boehmer, "z_Cowan": z_cowan, "t_date": t_date, "t_month": t_month}

In [ ]:
WINDOWS = {"[-1,+1]": ("car_1_win", 3), "[-5,+5]": ("car_5_win", 11), "[-10,+10]": ("car_win", 21)}
SIGNS = {"Negative": "neg", "Positive": "pos", "Neutral": "neu"}

rows = []
for name, s in SIGNS.items():
    for w, (col, width) in WINDOWS.items():
        rows.append({"Group": name, "Window": w, **group_tests(cars[cars["sign"] == s], col, width)})

pd.DataFrame(rows).set_index(["Group", "Window"]).round(2)


Size and sign: is the average in the direction the hypothesis predicts?
Agreement: do Boehmer, Cowan and the clustered tests say the same thing? If they don't, what's different about the clustered ones?
H2 specifically: a significantly negative result for positive campaigns doesn't just fail to support H2, it goes against it. Which window shows that?
Sample size: positive campaigns have 162 events vs 1,054 negative. How much weight can that carry?

## 5. Splits (H3, H4, H5)
*To do.* Use `car_table` within negative events, because prominence and reach should push negative and positive campaigns in opposite directions. Show the positive events as well, but note the small N. Variables: `mass` or `n_targets`, `prominence`, `ngo_power`, `climate` × `post`.

In [ ]:
neg = cars[cars["sign"] == "neg"]
print(neg["n_targets"].describe())
print((neg["n_targets"] == 1).sum(), neg["n_targets"].between(2, 10).sum(), neg["mass"].sum())

In [ ]:
def split_table(groups, col='car_win', width=21):
    rows = {name: group_tests(d, col, width) for name, d in groups.items()}
    return pd.DataFrame(rows).T.round(2)

split_table({
    "One firm": neg[neg["n_targets"] == 1],
    "2 to 10 firms": neg[neg["n_targets"].between(2, 10)],
    "More than 10 firms": neg[neg["mass"]],
})

In [ ]:
d_neg = neg.assign(y=100*neg["car_win"], x=neg["mass"].astype(int))
groups = np.column_stack([pd.factorize(d_neg["trade_date"])[0], pd.factorize(d_neg["ds_name"])[0]])
f = smf.ols("y ~ x", data=d_neg).fit(cov_type="cluster", cov_kwds={"groups": groups})
print(f.params["x"], f.tvalues["x"], f.pvalues["x"])

In [ ]:
pos = cars[cars["sign"] == "pos"]
print(pos["n_targets"].describe())
print((pos["n_targets"] == 1).sum(), pos["n_targets"].between(2, 10).sum(), pos["mass"].sum())

split_table({
    "One firm": pos[pos["n_targets"] == 1],
    "2 to 10 firms": pos[pos["n_targets"].between(2, 10)],
    "More than 10 firms": pos[pos["mass"]],
})

In [ ]:
d_pos = pos.assign(y=100*pos["car_win"], x=pos["mass"].astype(int))
groups = np.column_stack([pd.factorize(d_pos["trade_date"])[0], pd.factorize(d_pos["ds_name"])[0]])
f = smf.ols("y ~ x", data=d_pos).fit(cov_type="cluster", cov_kwds={"groups": groups})
print(f.params["x"], f.tvalues["x"], f.pvalues["x"])

Work through these questions, then send me your draft:

Direction: do fewer-target campaigns look worse than mass campaigns?
Evidence: is the gap statistically significant in Step 4?
The hypothesis itself: the one-firm and 2–10 means are almost the same. Does that fit "bank-specific matters more", or does it point to something narrower?
Mean vs median: the mass group has a mean of +0.20, but check its median in your table. What does the difference tell you about relying on the mean?
Sample size: how much weight can 86 events carry?

In [ ]:
print(neg["prominence"].value_counts().sort_index())
print(neg["ngo_power"].round(2).value_counts().sort_index())

In [ ]:
def diff_test(d,x):
    d = d.assign(y=100*d["car_win"], x = x.astype(int))
    groups = np.column_stack([pd.factorize(d["trade_date"])[0], pd.factorize(d["ds_name"])[0]])
    f = smf.ols("y ~ x", data = d).fit(cov_type="cluster", cov_kwds={"groups": groups})
    return {"Gap (pts)": f.params["x"], "t-value": f.tvalues["x"], "p-value": f.pvalues["x"]}

split_table({
    "Headline": neg[neg["prominence"] == 4],
    "First paragraph": neg[neg["prominence"] == 3],
    "Elsewhere or report only": neg[neg["prominence"] <= 2],
})

In [ ]:
diff_test(neg, neg["prominence"] == 4)

In [ ]:
split_table({
    "High Reach (2.5+)": neg[neg["ngo_power"] >= 2.5],
    "Low Reach": neg[neg["ngo_power"] < 2.5],
})

In [ ]:
diff_test(neg, neg["ngo_power"] >= 2.5)

Now REPEAT FOR POSITIVE

In [ ]:
print(pos["prominence"].value_counts().sort_index())
print(pos["ngo_power"].round(2).value_counts().sort_index())

In [ ]:
split_table({
    "Headline": pos[pos["prominence"] == 4],
    "First paragraph": pos[pos["prominence"] == 3],
    "Elsewhere or report only": pos[pos["prominence"] <= 2],
})

In [ ]:
diff_test(pos, pos["prominence"] == 4)

In [ ]:
split_table({
    "High Reach (2.5+)": pos[pos["ngo_power"] >= 2.5],
    "Low Reach": pos[pos["ngo_power"] < 2.5],
})

In [ ]:
diff_test(pos, pos["ngo_power"] >= 2.5)

Treat prominence and reach separately:

Prominence direction: do headline campaigns look worse than the rest? Does the effect fall steadily as prominence drops, or does the first-paragraph group break the pattern?
Prominence evidence: the headline group's tests are strong on its own, but what does the gap test say? Which of those two answers H4?
Reach direction: does the result go the way H4 predicts? If not, say so plainly, because a result against your hypothesis is still a finding.
Reach, mean vs median: high-reach campaigns have a mean near zero but a median of −1.03%, and Boehmer and Cowan disagree. What might a few large positive CARs be doing to that mean?
Multiple tests: you've now run several comparisons, so how much weight can one p-value of 0.10 carry?

In [ ]:
print(pd.crosstab(neg["climate"], neg["post"]))

In [ ]:
split_table({
    "Climate before COP21": neg[neg["climate"] & ~neg["post"]],
    "Climate after COP21": neg[neg["climate"] & neg["post"]],
    "Other issues before COP21": neg[~neg["climate"] & ~neg["post"]],
    "Other issues after COP21": neg[~neg["climate"] & neg["post"]],
})

In [ ]:
m = 100 * neg.groupby(["climate", "post"])["car_win"].mean()

climate_change = m.loc[True, True] - m.loc[True, False]
other_change = m.loc[False, True] - m.loc[False, False]

print(round(climate_change, 2), round(other_change, 2), round(climate_change - other_change, 2))

In [ ]:
d = neg.assign(y=100*neg["car_win"], climate=neg["climate"].astype(int), post=neg["post"].astype(int))
groups = np.column_stack([pd.factorize(d["trade_date"])[0], pd.factorize(d["ds_name"])[0]])
f = smf.ols("y ~ climate * post", data = d).fit(cov_type = "cluster", cov_kwds={"groups": groups})
pd.DataFrame({"Coefficients": f.params, "t-values": f.tvalues, "p-values": f.pvalues}).round(3)

In [ ]:
split_table({
    "Climate before COP21": pos[pos["climate"] & ~pos["post"]],
    "Climate after COP21": pos[pos["climate"] & pos["post"]],
    "Other issues before COP21": pos[~pos["climate"] & ~pos["post"]],
    "Other issues after COP21": pos[~pos["climate"] & pos["post"]],
})

In [ ]:
d = pos.assign(y=100*pos["car_win"], climate=pos["climate"].astype(int), post=pos["post"].astype(int))
groups = np.column_stack([pd.factorize(d["trade_date"])[0], pd.factorize(d["ds_name"])[0]])
f = smf.ols("y ~ climate * post", data = d).fit(cov_type = "cluster", cov_kwds={"groups": groups})
pd.DataFrame({"Coefficients": f.params, "t-values": f.tvalues, "p-values": f.pvalues}).round(3)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9, 4), sharey=True)
for ax, d, title in zip(axes, [neg, pos], ["Negative campaigns", "Positive campaigns"]):
    for climate, label, col, ls in [(True, "Climate", "#c8283c", "-"), (False, "Other issues", "#0d3b78", "--")]:
        m = 100 * d[d["climate"] == climate].groupby("post")["car_win"].mean()
        ax.plot([0, 1], m.to_numpy(), color=col, ls=ls, lw=2, marker="o", label=label)
    ax.axhline(0, color="#b7b5ad", lw=1)
    ax.set_xticks([0, 1], ["Before COP21", "After COP21"])
    ax.set_xlim(-0.2, 1.2)
    ax.set_title(title, loc="left", fontsize=11)
    ax.spines[["top", "right"]].set_visible(False)
axes[0].set_ylabel("Mean CAR over [-10, +10] (%)")
axes[0].legend(frameon=False)
fig.tight_layout()
plt.show()


In [ ]:
TRUMP = pd.Timestamp("2016-11-08")                           # Trump election

def trump_groups(d):
    after = d["trade_date"] > TRUMP
    return{
        "Climate before election": d[d["climate"] & ~after],
        "Climate after election": d[d["climate"] & after],
        "Other issues before election": d[~d["climate"] & ~after],
        "Other issues after election": d[~d["climate"] & after]
    }

split_table(trump_groups(neg))

In [ ]:
def trump_did(d):
    d = d.assign(y = 100*d["car_win"],
                 climate = d["climate"].astype(int),
                 post_trump = (d["trade_date"] > TRUMP).astype(int))
    groups = np.column_stack([pd.factorize(d["trade_date"])[0], pd.factorize(d["ds_name"])[0]])
    f = smf.ols("y ~ climate * post_trump", data = d).fit(cov_type = "cluster", cov_kwds={"groups": groups})
    return pd.DataFrame({"Coefficients": f.params, "t-values": f.tvalues, "p-values": f.pvalues}).round(3)

In [ ]:
trump_did(neg)

In [ ]:
split_table(trump_groups(pos))

In [ ]:
trump_did(pos)

In [ ]:
def mean_ci(d, col="car_win"):
    f = sm.OLS(100 * d[col].to_numpy(), np.ones(len(d))).fit(cov_type="cluster", cov_kwds={"groups": pd.factorize(d["trade_date"])[0]})
    return f.params[0], 1.96 * f.bse[0], len(d)

TRUMP = pd.Timestamp("2016-11-08")

fig, axes = plt.subplots(1, 2, figsize=(10, 4.5), sharey=True)
for ax, d, title in zip(axes, [neg, pos], ["Negative campaigns", "Positive campaigns"]):
    after = d["trade_date"] > TRUMP
    for climate, label, col, ls, shift in [(True, "Climate", "#c8283c", "-", -0.1), (False, "Other issues", "#0d3b78", "--", 0.1)]:
        groups = [d[(d["climate"] == climate) & ~after], d[(d["climate"] == climate) & after]]
        m, ci, n = zip(*[mean_ci(g) for g in groups])
        x = np.array([0, 1]) + shift
        ax.errorbar(x, m, yerr=ci, color=col, ls=ls, lw=2, marker="o", ms=6, elinewidth=1, label=label)
        for xi, top, ni in zip(x, np.array(m) + np.array(ci), n):
            ax.annotate(f"N = {ni}", (xi, top), xytext=(0, 3), textcoords="offset points",
                        ha="center", va="bottom", fontsize=8, color="#3a3936")
    ax.axhline(0, color="#b7b5ad", lw=1)
    ax.set_xticks([0, 1], ["Before the election", "After the election"])
    ax.set_xlim(-0.45, 1.45)
    ax.set_title(title, loc="left", fontsize=11)
    ax.grid(axis="y", color="#e8e6df", lw=0.8)
    ax.set_axisbelow(True)
    ax.spines[["top", "right"]].set_visible(False)
axes[0].set_ylabel("Mean CAR over [-10, +10] (%), with 95% CI")
axes[0].legend(frameon=False, loc="lower left")
fig.suptitle("Before and after the 2016 US election: the climate gap does not change significantly", x=0.07, ha="left", fontsize=12)
fig.tight_layout()
plt.show()


The test: is climate:post_trump significant? What does that mean for a climate-specific Trump effect?
The surprise: other-issue campaigns fall to −1.27%. You didn't predict this, and you've run many splits, so how much weight should it carry?
The limit: COP21 and the election are only 11 months apart. What does that stop you from claiming?

In [ ]:
def mean_ci(d, col="car_win"):
    f = sm.OLS(100 * d[col].to_numpy(), np.ones(len(d))).fit(cov_type="cluster", cov_kwds={"groups": pd.factorize(d["trade_date"])[0]})
    return f.params[0], 1.96 * f.bse[0], len(d)

FOREST = {
    "All negative campaigns": neg,
    "One firm named": neg[neg["n_targets"] == 1],
    "2 to 10 firms": neg[neg["n_targets"].between(2, 10)],
    "More than 10 firms": neg[neg["mass"]],
    "Headline mention": neg[neg["prominence"] == 4],
    "First paragraph": neg[neg["prominence"] == 3],
    "Elsewhere or report only": neg[neg["prominence"] <= 2],
    "High-reach NGO (2.5+)": neg[neg["ngo_power"] >= 2.5],
    "Lower-reach NGO": neg[neg["ngo_power"] < 2.5],
    "Climate, before COP21": neg[neg["climate"] & ~neg["post"]],
    "Climate, after COP21": neg[neg["climate"] & neg["post"]],
    "Other issues, before COP21": neg[~neg["climate"] & ~neg["post"]],
    "Other issues, after COP21": neg[~neg["climate"] & neg["post"]],
}
res = pd.DataFrame({k: mean_ci(d) for k, d in FOREST.items()}, index=["mean", "ci", "n"]).T
y = np.arange(len(res))[::-1]

fig, ax = plt.subplots(figsize=(9, 6))
ax.axvline(0, color="#b7b5ad", lw=1)
ax.errorbar(res["mean"], y, xerr=res["ci"], fmt="o", color="#c8283c", ms=6, lw=1.5, capsize=0)
ax.set_yticks(y, [f"{k}  (N = {int(n):,})" for k, n in zip(res.index, res["n"])])
ax.set_xlabel("Mean CAR over [-10, +10] (%), with 95% confidence interval clustered by date")
ax.set_title("Only headline and 2-to-10-firm campaigns sit clearly below zero", loc="left", fontsize=12)
ax.grid(axis="x", color="#e8e6df", lw=0.8)
ax.set_axisbelow(True)
ax.spines[["top", "right", "left"]].set_visible(False)
ax.tick_params(axis="y", length=0)
fig.tight_layout()
plt.show()


**Comment:** *(write after checking the table: size of the effect, its sign and which tests agree)*

## 6. Regression
*To do.* Regress `carw_[-10,+10]` on the campaign characteristics, with errors clustered by date and by bank. This is the test of whether the groups in section 5 really differ. Include `climate` and `post` as well as their interaction, and run it on negative events as well as pooled.


In [ ]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf
from pathlib import Path

HANDOFF = OUT
COP21 = pd.Timestamp("2015-12-12")

cars = pd.read_csv(HANDOFF / "cars.csv", parse_dates=["trade_date"])
cars["post"] = cars["trade_date"] > COP21
cars["month"] = cars["trade_date"].dt.to_period("M").astype(str)

for col in ["car", "car_1", "car_5", "car_ff"]:
    lo, hi = cars[col].quantile([0.01, 0.99])
    cars[col + "_win"] = cars[col].clip(lo, hi)

neg = cars[cars["sign"] == "neg"]
pos = cars[cars["sign"] == "pos"]

RHS = "sentiment + climate * post + prominence + ngo_power + mass + region"

def run_reg(col, d):
    d = d.assign(y=100 * d[col], climate=d["climate"].astype(int), post=d["post"].astype(int), mass=d["mass"].astype(int))
    groups = np.column_stack([pd.factorize(d["trade_date"])[0], pd.factorize(d["ds_name"])[0]])
    return smf.ols(f"y ~ {RHS}", data=d).fit(cov_type="cluster", cov_kwds={"groups": groups})


In [ ]:
f = run_reg("car_win", neg)
pd.DataFrame({"Coefficients": f.params, "t-values": f.tvalues, "p-values": f.pvalues}).round(3)

In [ ]:
SPECS = {
    "(1) All, [-10,+10]": ("car_win", cars),
    "(2) Negative [-10,+10]": ("car_win", neg),
    "(3) Negative [-5,+5]": ("car_5_win", neg),
    "(4) Negative, Fama-French": ("car_ff_win", neg),
    "(5) Positive, [-10,+10]": ("car_win", pos),
}

fits = {name: run_reg(col, d) for name, (col, d) in SPECS.items()}

reg = pd.DataFrame({name: {v: f"{f.params[v]:.3f} ({f.tvalues[v]:.3f})" for v in f.params.index} 
                   for name, f in fits.items()})

reg.loc["N"] = {name: f"{int(f.nobs):,}" for name, f in fits.items()}
reg.loc["R-squared"] = {name: f"{f.rsquared:.3f}" for name, f in fits.items()}
reg

The main test: are any of prominence, ngo_power, mass or climate:post significant in column (2)? What does that mean for H3, H4 and H5?
Splits vs regression: the headline group looked strong in H4. Does that survive here? prominence assumes each step from 1 to 4 has the same effect, but your H4 table wasn't a steady pattern. Try swapping in a headline dummy to see if it changes your answer.
Stability: do the signs stay the same across columns (2), (3) and (4)? prominence and climate:post are worth a look.
post in column (4): its t of −1.97 is the only one near significance in this table. With this many coefficients, how much weight should one carry?
R²: it's under 1% in the negative columns. What does that say about how much these characteristics explain?
Pooled vs negative: why is column (2) your main result and not column (1)?


## 7. Main figure
*To do.* Average cumulative AR from `ar_path`, day −10 to +10, negative against positive, with confidence bands.

In [ ]:
import matplotlib.pyplot as plt

t0 = cars.merge(panel[["ds_name", "date", "t"]], left_on=["ds_name", "trade_date"],
                right_on=["ds_name", "date"], how="left", validate="1:1")["t"]
print(t0.isna().sum())


In [ ]:
params = betas.set_index(["ds_name", "t"])[["capm_alpha", "capm_mkt_rf"]]
by_bank = {name: d.set_index("t") for name, d in panel.groupby("ds_name")}

paths = []
for name, t in zip(cars["ds_name"], t0.astype(int)):
    win = by_bank[name].loc[t - PRE:t + POST]
    alpha, beta = params.loc[(name, t)]
    paths.append((win["exret"] - (alpha + beta * win["mkt_rf"])).to_numpy())

ar = np.vstack(paths)
print(ar.shape)
print(np.abs(ar.sum(axis=1) - cars["car"]).max())


In [ ]:
days = np.arange(-PRE, POST + 1)
cum = ar.cumsum(axis=1)
lo, hi = np.quantile(cum, [0.01, 0.99], axis=0)
cum_w = np.clip(cum, lo, hi)


In [ ]:
def path_stats(mask):
    dates = pd.factorize(cars.loc[mask, "trade_date"])[0]
    fits = [sm.OLS(100 * cum_w[mask, j], np.ones(mask.sum())).fit(cov_type="cluster", cov_kwds={"groups": dates})
            for j in range(len(days))]
    return np.array([f.params[0] for f in fits]), np.array([f.bse[0] for f in fits])

STYLE = {"neg": ("Negative campaigns", "#c8283c", "-"), "pos": ("Positive campaigns", "#0d3b78", "--")}
path = {s: path_stats((cars["sign"] == s).to_numpy()) for s in STYLE}

pd.DataFrame({STYLE[s][0]: {f"Day {d:+d}": path[s][0][d + PRE] for d in (-10, -5, -1, 0, 1, 5, 10)} for s in STYLE}).round(2).T

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
ax.axhline(0, color="#b7b5ad", lw=1)
ax.axvline(0, color="#b7b5ad", lw=1)

for s, (label, col, ls) in STYLE.items():
    m, se = path[s]
    n = (cars["sign"] == s).sum()
    ax.fill_between(days, m - 1.96 * se, m + 1.96 * se, color=col, alpha=0.13, lw=0)
    ax.plot(days, m, color=col, ls=ls, lw=2, label=f"{label} (N = {n:,})")
    ax.annotate(f"{label.split()[0]}: {m[-1]:.2f}%", (days[-1], m[-1]), xytext=(8, 0),
                textcoords="offset points", va="center", fontsize=10, color="#3a3936")

ax.set_xlim(-10, 10)
ax.set_xticks(range(-10, 11, 2))
ax.set_xlabel("Trading days relative to the campaign")
ax.set_ylabel("Cumulative abnormal return (%)")
ax.set_title("Bank stocks slip after NGO campaigns of either tone, but the bands include zero", loc="left", fontsize=12)
ax.grid(axis="y", color="#e8e6df", lw=0.8)
ax.set_axisbelow(True)
ax.spines[["top", "right"]].set_visible(False)
ax.legend(frameon=False, loc="lower left")
fig.subplots_adjust(right=0.82)
plt.show()


Timing: is there a drift before day 0? What would a large drop before the campaign have suggested?
When does it move? Negative is still +0.05% on day 0 and −0.10% on day +1. Sigwatch's date is when the action was entered into its database, so how might that affect your reading?
Size vs certainty: do the bands include zero at day +10? How does that fit with your t-statistics?
Negative vs positive: both lines fall and the bands overlap. What does that mean for the idea that the market responds to a campaign's tone?
Slide wording: this is your main slide, so what one sentence should a board member take from it?




## 8. Robustness
*To do.* Fama-French CARs (`car_ff` in `cars.csv`), other gaps (`build_sample(events, gap=...)`), β = 1, and dropping stale-price banks.

In [ ]:
t0 = cars.merge(panel[["ds_name", "date", "t"]], left_on=["ds_name", "trade_date"],
                right_on=["ds_name", "date"], how="left", validate="1:1")["t"]
by_bank = {name: d.set_index("t") for name, d in panel.groupby("ds_name")}

car_mkt = []
for name, t in zip(cars["ds_name"], t0.astype(int)):
    win = by_bank[name].loc[t - PRE:t + POST]
    car_mkt.append((win["exret"] - win["mkt_rf"]).sum())
cars["car_mkt"] = car_mkt

lo, hi = cars["car_mkt"].quantile([0.01, 0.99])
cars["car_mkt_win"] = cars["car_mkt"].clip(lo, hi)

print(100 * cars.loc[cars["sign"] == "neg", "car_mkt_win"].mean())


In [ ]:
zero = panel[panel["date"] >= "2010-01-01"].assign(z=lambda d: d["ret_local"].abs() < 1e-12).groupby("ds_name")["z"].mean()
stale = zero[zero > 0.10].index
print(list(stale))

In [ ]:
GROUPS = {
    "All negative": cars["sign"] == "neg",
    "All positive": cars["sign"] == "pos",
    "Negative, headline": (cars["sign"] == "neg") & (cars["prominence"] == 4),
    "Negative, 10 firms or fewer": (cars["sign"] == "neg") & ~cars["mass"],
    "Negative, lower-reach NGO": (cars["sign"] == "neg") & (cars["ngo_power"] < 2.5),
    "Negative, climate after COP21": (cars["sign"] == "neg") & cars["climate"] & cars["post"],
}
CHECKS = {
    "Baseline": ("car_win", True),
    "Fama-French": ("car_ff_win", True),
    "Beta = 1": ("car_mkt_win", True),
    "Raw CARs": ("car", True),
    "Clean windows": ("car_win", cars["n_other"] == 0),
    "No stale banks": ("car_win", ~cars["ds_name"].isin(stale)),
}


In [ ]:
rows = []
for g, mask in GROUPS.items():
    for c, (col, keep) in CHECKS.items():
        d = cars[mask & keep]
        rows.append({"Group": g, "Check": c, "N": len(d), "Mean (%)": 100 * d[col].mean(),
                     "t date": clustered_t(d[col], d["trade_date"]), "t month": clustered_t(d[col], d["month"])})
robust = pd.DataFrame(rows)

def show(value):
    return robust.pivot(index="Group", columns="Check", values=value).reindex(index=list(GROUPS), columns=list(CHECKS)).round(2)

show("Mean (%)")


Which check matters most? Fama-French changes the results only a little: for negative campaigns on [−10,+10] the mean goes from −0.39% under CAPM to −0.33%, and for positive campaigns on [−1,+1] from −0.59% to −0.46%. What might HML, SMB, RMW and CMA be picking up that the market factor misses, and why do they change so little?

Your strongest result: headline campaigns are the most stable. Under which checks does it fall below 1.96? And remember the gap test in section 5 wasn't significant.
Clean windows: do means get larger or smaller when you remove other campaigns? What does that suggest about the 45% of events with another campaign nearby? Note the N falls to 115 for headline.
Other gaps: cite Nico's overlap-gap table for negative vs positive instead of repeating it. His Fama-French five-factor negative results stay insignificant at every gap.



## 9. D variable
*To do.* One more split once the media or political variable exists, keyed on `uid_archive` through `uids`.

In [ ]:
media = pd.read_csv(HANDOFF / "media_coverage.csv")   # replace with D's file; columns: uid_archive, media
media["uid_archive"] = media["uid_archive"].astype(str)

links = cars[["ds_name", "trade_date", "uids"]].assign(uid_archive=cars["uids"].astype(str).str.split(";")).explode("uid_archive")
links = links.merge(media, on="uid_archive", how="left")
print(links["media"].isna().mean())                   # share of campaigns with no match

event_media = links.groupby(["ds_name", "trade_date"], as_index=False)["media"].max()
cars = cars.merge(event_media, on=["ds_name", "trade_date"], how="left", validate="1:1")

neg = cars[cars["sign"] == "neg"]
split_table({"Major coverage": neg[neg["media"] == 1], "No major coverage": neg[neg["media"] == 0]})
